In [ ]:
## loading packages
import pandas as pd
import numpy as np
import plotly.graph_objects as go
#import plotly.express as px
import argparse
import sys
import re
import random
import difflib
from collections import defaultdict
import copy
import json
import os

import logging

# Configure once (usually at program start)
logging.basicConfig(
    level=logging.INFO,  # minimum level to display
    format="%(asctime)s [%(levelname)s] %(message)s"
)

def is_notebook():
    try:
        from IPython import get_ipython
        shell = get_ipython().__class__.__name__
        if shell == 'ZMQInteractiveShell':
            return True   # Jupyter notebook or qtconsole
        elif shell == 'TerminalInteractiveShell':
            return False  # Terminal running IPython
        else:
            return False  # Other type (?)
    except Exception:
        return False      # Probably standard Python interpreter

   
def float_0_1(value):
    f = float(value)
    if f < 0 or f > 1:
        raise argparse.ArgumentTypeError(f"{value} is not in range [0, 1]")
    return f

In [ ]:
## argparse definition
if not is_notebook():
    script_name = sys.argv[0]
else:
    script_name = 'Script'

parser = argparse.ArgumentParser(description=f"{script_name} parameters")

## eigenvectors
parser.add_argument('--eigenvec', type=str, default=None, required=False, help='Eigenvec file path (plink)')

parser.add_argument('--eigenvecID', type=str, default=None, help='Eigenvec ID column (default first column)')

parser.add_argument('--selectedID', type=str, default=None, required=False, help='Selected samples, given as comma separated IDs, or as a file with each ID on a line (default: all))')

## eigenvalues
parser.add_argument('--eigenval', type=str, default=None, required=False, help='Eigenval file path (plink)')
parser.add_argument('--nb_eigenvalues', type=int, default=10, help='Number of eigenvalues to show (0 for all)')

## stats
parser.add_argument('--imiss', type=str, default=None, help='Imiss file path (plink)')
parser.add_argument('--lmiss', type=str, default=None, help='Lmiss file path (plink)')
parser.add_argument('--frq', type=str, default=None, help='Frq file path (plink)')

## annotation
parser.add_argument('--annotation', type=str, default=None, help='Annotation file path')
parser.add_argument('--annotationID', type=str, default='Genetic ID', help='Annotation ID column (default first column)')
parser.add_argument('--longitude', type=str, default=None, help='Longitude column name')
parser.add_argument('--latitude', type=str, default=None, help='Latitude column name')
parser.add_argument('--time', type=str, default=None, help='Time column name')
parser.add_argument('--group', type=str, default=None, help='Grouping/coloring column name')

## text handling
parser.add_argument('--ignore_case', action='store_true', default=False, help='Ignore case differences in annotation')
parser.add_argument('--ignore_space', action='store_true', default=False, help='Ignore space differences in annotation')

parser.add_argument('--col_abbrev', type=int, default=15, help='Abbreviate column names to this length (0 for no abbreviation)')
parser.add_argument('--legend_abbrev', type=int, default=0, help='Abbreviate legend text to this length (0 for no abbreviation)')

parser.add_argument('--max_factors', type=int, default=400, help='Maximum number of different elements in factorial columns to be included')

## aesthetics
parser.add_argument('--aesthetics_file', type=str, default=None, help='Json file with stored aesthetics (default none)')

parser.add_argument('--color_schema_continuous', type=str, default='Viridis', help='Color schema for continuous variables (default Viridis)')

parser.add_argument('--point_color', type=str, default="#000000", help='Default point color for selected points (default #000000)')
parser.add_argument('--point_color_unselected', type=str, default='#cccccc', help='Default point color for unselected points (default #cccccc)')
parser.add_argument('--point_size', type=int, default=8, help='Default point size (default 8)')
parser.add_argument('--point_size_unselected', type=int, default=8, help='Default point size for unselected points (default 8)')
parser.add_argument('--point_opacity', type=float_0_1, default=0.9, help='Default opacity (default 0.9)')
parser.add_argument('--point_opacity_unselected', type=float_0_1, default=0.3, help='Default opacity for unselected points (default 0.3)')
parser.add_argument('--point_symbol', type=str, default='circle', help='Default point symbol (default circle)')
parser.add_argument('--point_symbol_unselected', type=str, default='circle', help='Default point symbol for unselected points (default circle)')

## time figure
parser.add_argument('--time_plot_type', type=int, choices=[0, 1, 2], default=0, help='Time plot type: 0=scatter, 1=histogram with selection, 2=histogram simple')
parser.add_argument('--time_hist_nbins', type=int, default=100, help='Number of bins for the time histogram (100)')
parser.add_argument('--time_invert', action='store_true', default=False, help='Invert time axis (for BP data)')

## plot settings
parser.add_argument('--hover_minimal', action='store_true', default=False, help='Show minimal information when hovering points')
parser.add_argument('--open_browser', action='store_true', default=False, help='Open directly the dash server in a web browser')
parser.add_argument('--server_port', type=int, default=8050, help='Port for the dash server (default 8050)')

## development
parser.add_argument('--dev', action='store_true', default=False, help='Use development parameters')
parser.add_argument('--show_all_legends', action='store_true', default=False, help='Show the legends in all figures')


#parser.print_help()

In [ ]:

## development arguments
# Parse NOTHING when running in a notebook
if is_notebook():
    args = parser.parse_args([])      # <— key line ([])
else:
    args = parser.parse_args()

if is_notebook() or args.dev:
    args.eigenvec='data/aadr.eigenvec'

  
    args.eigenval='data/aadr.eigenval'

    args.imiss='data/aadr.imiss'
    args.lmiss='data/aadr.lmiss'
    args.frq='data/1000gp.frq'

    args.annotation='data/aadr.anno'
    
    args.reduction='MDS'
    args.eigenvecID='sample'
    args.annotationID='Genetic ID'
    args.nb_eigenvalues=10
    args.longitude='Long.'
    args.latitude='Lat.'
    args.time='zDate mean in BP in years before 1950 CE [OxCal mu for a direct radiocarbon date, and average of range for a contextual date]'
    args.group='Political_Entit'
    args.group='Molecular_Sex'

    args.ignore_case=True
    args.ignore_space=True

    args.col_abbrev=15
    args.legend_abbrev=20

    args.max_factors=400

    args.open_browser=False
    args.server_port=8050
    args.time_hist_nbins=500
    args.time_invert = True

    #args.aesthetics_file='/Users/sneuensc/Downloads/data(1).json'

    args.show_all_legends = False
    
    args.dev = True

if not args.eigenvec:
    logging.error('Missing argument: --eigenvec  is required.')
    sys.exit(1)

if args.dev:
    logging.getLogger().setLevel(logging.DEBUG)
    logging.debug('Development mode activated.')
else:
    logging.getLogger().setLevel(logging.INFO)

In [ ]:
## functions

## abbreviate a list of strings to a maximum length, preserving uniqueness
def make_unique_abbr(cur_list, max_length=3):
    # Compile regex patterns once
    clean_re = re.compile(r'[^a-zA-Z0-9 ]')
    space_re = re.compile(r' ')

    # Clean: remove special characters 
    cleaned = [clean_re.sub('', str(elem)) for elem in cur_list]

    # Abbreviate: replace spaces with underscores and truncate
    abbreviated = [space_re.sub('_', elem[:max_length]).rstrip('_') for elem in cleaned]

    # Make unique using a counter
    counter = defaultdict(int)
    unique_abbr = []
    for abbr in abbreviated:
        new_abbr = abbr
        while new_abbr in counter:
            counter[abbr] += 1
            new_abbr = f"{abbr}{counter[abbr]}"
        counter[new_abbr] = 0
        unique_abbr.append(new_abbr)
        #print(f"{abbr} -> {new_abbr}")

    return unique_abbr


## abbreviate columns of a pandas DataFrame
def make_unique_abbr_of_df(df, cols, max_length=3):
    """
    Abbreviate the values in the specified columns of a DataFrame.
    Each unique value in the column is replaced by a unique abbreviation.
    """
    for col in cols:
        unique_vals = df[col].astype(str).unique()
        abbrs = make_unique_abbr(unique_vals, max_length)
        abbr_map = dict(zip(unique_vals, abbrs))
        df[col] = df[col].astype(str).map(abbr_map)
    return df


## search the the best text match within a list of words
def get_abbr_of(target, lookup, returns=None):
    closest = difflib.get_close_matches(target, lookup, n=1)
    if closest is None or len(closest) == 0:
        print(f"Warning: No match found for {target} in {lookup}")
        return None
    
    if returns is  None:
        match = closest[0]
    else:
        match = returns[lookup.index(closest[0])]

    #print(f"Look up for {target} => {match}")
    return match



## de-duplex columns ignoring capitalization and space differences (keep first version of each)
def deduplicate_columns(df, columns, ignore_case=True, ignore_space=True):
    for col in columns:
        norm_col = f'_norm_{col}'
        series = df[col].astype(str)
        if ignore_case:
            series = series.str.lower()
        if ignore_space:
            series = series.str.replace(r'\s+', '', regex=True)
        df[norm_col] = series
        first_map = df.drop_duplicates(norm_col, keep='first').set_index(norm_col)[col]
        df[col] = df[norm_col].map(first_map)
        df.drop(columns=[norm_col], inplace=True)
    return df

## find automatically the pca dimensions prefix
def find_incrementing_prefix_series(columns):
    # Match prefix + number, e.g., PC1, PC2, Dim3
    pattern = re.compile(r'^([A-Za-z_]+)(\d+)$')
    prefix_groups = defaultdict(list)

    # Group columns by prefix
    for col in columns:
        match = pattern.match(col)
        if match:
            prefix, num = match.groups()
            prefix_groups[prefix].append(int(num))

    # Find prefixes with longest incrementing series
    longest_series = []
    for prefix, nums in prefix_groups.items():
        nums_sorted = sorted(nums)
        # Check if numbers form a consecutive sequence
        if nums_sorted == list(range(nums_sorted[0], nums_sorted[-1] + 1)):
            series = [f"{prefix}{n}" for n in nums_sorted]
            if len(series) > len(longest_series):
                longest_series = series

    return longest_series


def dict_of_dicts_to_tuple(d):
    if d is None:
        return tuple()

    def _normalize(val):
        if isinstance(val, dict):
            return tuple(
                (str(k), _normalize(v))
                for k, v in sorted(val.items(), key=lambda it: str(it[0]))
            )
        if isinstance(val, (list, tuple)):
            return tuple(_normalize(v) for v in val)
        # keep primitives (int/float/str/bool/None). For other types fallback to str().
        if isinstance(val, (int, float, str, bool)) or val is None:
            return val
        return str(val)

    return tuple((str(k), _normalize(v)) for k, v in sorted(d.items(), key=lambda it: str(it[0])))


def tuple_to_dict_of_dicts(t):
    if not t:
        return {}

    def _denormalize(val):
        # tuple of key/value pairs -> dict
        if isinstance(val, tuple) and all(isinstance(e, tuple) and len(e) == 2 for e in val):
            return {k: _denormalize(v) for k, v in val}
        # other tuples are lists (preserve order)
        if isinstance(val, tuple):
            return [_denormalize(v) for v in val]
        return val

    return {k: _denormalize(v) for k, v in t}


## save aesthetics dict of dicts to json
def save_dict_of_dicts_to_json(data, filename):
    with open(filename, 'w') as f:
        json.dump(data, f, indent=2)


## read aesthetics dict of dicts from json
def read_dict_of_dicts_from_json(filename):
    import json
    with open(filename, 'r') as f:
        return json.load(f)

In [ ]:
## reading imiss
if args.imiss is not None:
    logging.info(f"Reading imiss file '{args.imiss}' ...")
    df_imiss = pd.read_csv(args.imiss, sep=r"\s+", skiprows=[1])
    logging.info(f"Reading imiss file '{args.imiss}' ... done.")
else:
    df_imiss = None

In [ ]:
## reading lmiss
if args.lmiss is not None:
    logging.info(f"Reading lmiss file '{args.lmiss}' ...")
    df_lmiss = pd.read_csv(args.lmiss, sep=r"\s+", skiprows=[1], low_memory=False)
    logging.info(f"Reading lmiss file '{args.lmiss}' ... done.")
else:
    df_lmiss = None

In [ ]:
## reading frq
if args.frq is not None:
    logging.info(f"Reading frq file '{args.frq}' ...")
    df_frq = pd.read_csv(args.frq, sep=r"\s+")
    logging.info(f"Reading frq file '{args.frq}' ... done.")
else:
    df_frq = None

In [ ]:
## reading eigenval

if args.eigenval is not None:
    logging.info(f"Reading eigenval file '{args.eigenval}' ...")
    eigenval = pd.read_csv(args.eigenval, sep="\t", header=None, names=["eigenvalue"])

    logging.info(f"   Found {len(eigenval)} eigenvalues.")

    ## compute variance explained
    eigenval["eigenvalue"] = eigenval["eigenvalue"] / eigenval["eigenvalue"].sum()

    ## add cumulative eigenvalues
    eigenval["cumulative"] = eigenval["eigenvalue"].cumsum()

    ## add index starting from 1 in the first column
    eigenval["dimension"] = eigenval.index + 1

    ## order columns
    eigenval = eigenval[["dimension"] + [col for col in eigenval.columns if col != "dimension"]]

    logging.info(f"Reading eigenval file '{args.eigenval}' ... done.")
else:
    eigenval = None



In [ ]:
## reading eigenvec (always present)

# Read eigenvectors
logging.info(f"Reading eigenvec file '{args.eigenvec}' ...")
eigenvec = pd.read_csv(args.eigenvec, sep=r"\s+", header=0)

PCS = find_incrementing_prefix_series(eigenvec.columns)

if len(PCS) < 2:
    logging.error('Not enough dimension columns found in the eigenvec file ({len(PCS)} found)).')

logging.info(f"   Found {len(PCS)} principal components ({", ".join(PCS[:2])}, ...).")

# Scale PC columns: to be removed
nb_snp = 194926 # Number of SNPs
eigenvec[PCS] = eigenvec[PCS].div(nb_snp)

logging.info(f"Reading eigenvec file '{args.eigenvec}' ... done.")

In [ ]:
## reading annotation
annotation = None
ANNOTATION_TIME = None
ANNOTATION_LAT = None
ANNOTATION_LONG = None
annotation_desc = None

if args.annotation is not None:
    logging.info(f"Reading annotation file '{args.annotation}' ...")
    annotation = pd.read_csv(args.annotation, sep='\t', na_values='..', low_memory=False)
    
    # create abbreviation_desc data.frame
    annotation_desc = pd.DataFrame({
        'Abbreviation': make_unique_abbr(annotation.columns, max_length=args.col_abbrev),
        'Description': annotation.columns,
        'Type': ['continuous' if annotation[col].dtype.kind in 'fi' else 'categorical' for col in annotation.columns],
        'N_levels': [annotation[col].nunique(dropna=False) if annotation[col].dtype.kind not in 'fi' else None for col in annotation.columns]
    })

    # Add 'Dropdown' column in one go
    annotation_desc['Dropdown'] = [
        'Yes' if ((typ == 'continuous') or ((typ == 'categorical') and (nlev <= args.max_factors))) else ''
        for typ, nlev in zip(annotation_desc['Type'], annotation_desc['N_levels'])
    ]
    

    # Rename columns of annotation data.frame with the abbreviations
    annotation.columns = annotation_desc['Abbreviation']

    # Get abbreviations for given IDs
    if args.annotationID is not None:
        ANNOTATION_ID = get_abbr_of(args.annotationID, annotation_desc['Description'].to_list(), annotation_desc['Abbreviation'].to_list())
        if ANNOTATION_ID not in annotation.columns:
            logging.info(f"Warning: Specified annotationID '{--annotationID}' not found in annotation columns. Using first column instead.")
    else:
        ANNOTATION_ID = annotation.columns[0] ## default is the first column

    if args.time is not None:
        ANNOTATION_TIME = get_abbr_of(args.time, annotation_desc['Description'].to_list(), annotation_desc['Abbreviation'].to_list())
        if ANNOTATION_TIME not in annotation.columns:
            ANNOTATION_TIME = None
            logging.warning(f"Warning: Specified time '{--time}' not found in annotation columns. Time graph disabled.")

    if args.longitude is not None:
        ANNOTATION_LONG = get_abbr_of(args.longitude, annotation_desc['Description'].to_list(), annotation_desc['Abbreviation'].to_list())
        if ANNOTATION_LONG not in annotation.columns:
            ANNOTATION_LONG = None
            logging.warning(f"Warning: Specified longitude '{--longitude}' not found in annotation columns. Geographical map disabled.")

    if args.latitude is not None:
        ANNOTATION_LAT = get_abbr_of(args.latitude, annotation_desc['Description'].to_list(), annotation_desc['Abbreviation'].to_list())
        if ANNOTATION_LAT not in annotation.columns:
            ANNOTATION_LAT = None
            logging.warning(f"Warning: Specified latitude '{--latitude}' not found in annotation columns. Geographical map disabled.")

    ## clean factorial elements if needed
    exclude_abbr = [elem for elem in [ANNOTATION_ID, ANNOTATION_TIME, ANNOTATION_LONG, ANNOTATION_LAT] if elem != None]

    ## define the columns adequate for grouping/coloring
    GROUPING_COLUMNS = annotation_desc.loc[
        (annotation_desc['Type'] == 'continuous') |
        ((annotation_desc['Type'] == 'categorical') & (annotation_desc['N_levels'] <= args.max_factors)) &
        (~annotation_desc['Abbreviation'].isin(exclude_abbr)),
        'Abbreviation'
    ].tolist()


    ## adjust/clean categorical columns used for grouping/coloring
    categorical2adjust = annotation_desc.loc[
        (annotation_desc['Dropdown'] == 'Yes') & 
        (annotation_desc['Type'] == 'categorical'),
        'Abbreviation'
    ].tolist()
    
    if len(categorical2adjust) > 0:
        if args.ignore_case or args.ignore_space:
            if args.ignore_case:
                logging.info(f"   Ignoring case differences.")
            if args.ignore_space:
                logging.info(f"   Ignoring space differences.")
            annotation = deduplicate_columns(annotation, categorical2adjust, args.ignore_case, args.ignore_space)

        if args.legend_abbrev > 0:
            logging.info(f"   Shortening legend text to max {args.legend_abbrev} characters.")
            annotation = make_unique_abbr_of_df(annotation, categorical2adjust, args.legend_abbrev)

    ## logging
    logging.info(f"   Found {(annotation_desc['Type'] == 'categorical').sum()} categorical columns in total.")
    logging.info(f"   Found {len(categorical2adjust)} categorical columns used for grouping/coloring.")
    logging.info(f"   Found {(annotation_desc['Type'] == 'continuous').sum()} continuous columns.")
    if ANNOTATION_LAT and ANNOTATION_LONG:
        logging.info(f"   Found geographic coordinates columns (latitude/longitude): '{ANNOTATION_LAT}', '{ANNOTATION_LONG}'.")
    if ANNOTATION_TIME:
        logging.info(f"   Found time column: '{ANNOTATION_TIME}'.")
    
    logging.info(f"Reading annotation file '{args.annotation}' ... done.")

In [ ]:
## merging coord and annotation

# Check if annotation exists (not None and not empty)
if annotation is not None and not annotation.empty:
    EIGENVEC_ID = get_abbr_of(args.eigenvecID, eigenvec.columns.to_list()) if args.eigenvecID is not None else eigenvec.columns[0] ## default is the first column

    # Rename EIGENVEC_ID column in annotation to match EIGENVEC_ID in eigenvec
    # so we can join by that column name
    annotation_renamed = annotation.rename(columns={ANNOTATION_ID: EIGENVEC_ID})
    
    # Perform left join
    coord = eigenvec.merge(annotation_renamed, on=EIGENVEC_ID, how="left")
    
    # Negate the values in the ANNOTATION_TIME column
    if args.time_invert and ANNOTATION_TIME is not None:
        coord[ANNOTATION_TIME] = -coord[ANNOTATION_TIME]
else:
    EIGENVEC_ID = args.eigenvecID if args.eigenvecID is not None else eigenvec.columns[0] ## default is the first column
    coord = eigenvec

# Rename the EIGENVEC_ID column to id
coord = coord.rename(columns={EIGENVEC_ID: "id"})

# Move 'id' column to the front
cols = ['id'] + [c for c in coord.columns if c != 'id']
coord = coord[cols]

In [ ]:
## init variables

##----------------------------------------------------
df = coord  # Assumed to be preloaded

##----------------------------------------------------

def get_annotation_table():
    pcs_table = pd.DataFrame({
        'Abbreviation': PCS,
        'Description': [f'Principal component {pcs}' for pcs in PCS],
        'Type': ['continuous'] * len(PCS),
        'N_levels': [None] * len(PCS),
        'Dropdown': ['Yes'] * len(PCS),
    })
    pcs_table['N_levels'] = pcs_table['N_levels'].astype('float64')

    if annotation_desc is not None:
        annotation_desc['N_levels'] = annotation_desc['N_levels'].astype('float64', errors='ignore')

        tmp = pd.concat([annotation_desc, pcs_table], ignore_index=True)
        tmp.loc[tmp['Abbreviation'] == ANNOTATION_ID, 'Abbreviation'] = 'id'
        return tmp

    return pcs_table


annotation_desc_ext= get_annotation_table()


In [ ]:


##----------------------------------------------------
import dash
from dash import dcc, html, Input, Output, State
from dash import callback_context as ctx
from dash_ag_grid import AgGrid
import plotly.express as px
import pandas as pd
from functools import lru_cache
import time
from functools import wraps
import webbrowser
import dash_daq as daq
from dash.dependencies import ALL
import dash_bootstrap_components as dbc
from dash.exceptions import PreventUpdate




init_x = PCS[0]
init_y = PCS[1]
init_z = PCS[2]

plotly_point_sizes = np.arange(4, 18, 2)
plotly_opacity_sizes = np.arange(0, 1.1, 0.1)
print(plotly_opacity_sizes)

## selection
if args.selectedID:
    if os.path.isfile(args.selectedID):
        ## read the file, line by line
        with open(args.selectedID, 'r') as f:
            init_selected_ids = [line.rstrip('\n') for line in f]

        total = len(df['id'].tolist())
        init_unique = list(set(init_selected_ids) & set(total))
        logging.info(f"{len(init_unique)} of {len(total)} samples selected (IDs specified by a file: {round(100*len(init_unique)/len(init_selected_ids))} of the given samples are valid)")
    else:
        init_selected_ids = args.selectedID.split(";")
        total = len(df['id'].tolist())
        init_unique = list(set(init_selected_ids) & set(total))
        logging.info(f"{len(init_unique)} of {len(total)} samples selected (IDs specified by a string: {round(100*len(init_unique)/len(init_selected_ids))} of the given samples are valid)")
        init_selected_ids = init_unique
else:
    init_selected_ids = df['id'].tolist()
    logging.info(f"All {len(init_selected_ids)} samples selected (default)")


## group/color
dropdown_group_list = ['none']
if args.annotation:
    dropdown_group_list += annotation_desc_ext.loc[annotation_desc_ext['Dropdown'] == 'Yes', 'Abbreviation'].tolist()

init_group = args.group if args.group is not None and args.group in dropdown_group_list else dropdown_group_list[0]


## time / continuous variable
if args.annotation:
    dropdown_list_continuous = annotation_desc_ext.loc[annotation_desc_ext['Type'] == 'continuous', 'Abbreviation'].tolist()
    if dropdown_list_continuous:
        init_continuous = ANNOTATION_TIME if ANNOTATION_TIME is not None and ANNOTATION_TIME in dropdown_list_continuous else dropdown_list_continuous[0]
    else:
        init_continuous = None
else:
    dropdown_list_continuous = None
    init_continuous = None




symbols_list = list(range(1, 11))

plotly_point_symbols = [
    "circle", "circle-open",
    "square", "square-open",
    "diamond", "diamond-open",
    "cross", "x",
    "triangle-up", "triangle-down",
    "triangle-left", "triangle-right",
    "star", "star-open"
]

maplot_point_symbols = ['circle', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star']


##----------------------------------------------------
## get the selected and unselected dataframe based on selected ids or just the selected_df (Use numpy for fast masking)
def get_selected_df(selected_ids, df_=None):
    df_use = df if df_ is None else df_
    if not selected_ids:
        return df_use
    selected_mask = np.isin(df_use['id'], selected_ids)
    assert(len(selected_ids) == sum(selected_mask))
    return df_use[selected_mask]

def get_unselected_df(selected_ids, df_=None):
    df_use = df if df_ is None else df_
    if not selected_ids:
        return df_use.iloc[0:0]  # empty DataFrame with same columns
    selected_mask = np.isin(df_use['id'], selected_ids)
    assert(len(selected_ids) == sum(selected_mask))
    return df_use[~selected_mask]

def get_selected_df_both(selected_ids, df_=None):
    df_use = df if df_ is None else df_
    if not selected_ids:
        return df_use, df_use.iloc[0:0]
    selected_mask = np.isin(df_use['id'], selected_ids)
    assert(len(selected_ids) == sum(selected_mask))
    return df_use[selected_mask], df_use[~selected_mask]

##---------------------------------------------------
# ## time logger for callbacks
def log_time(name=None):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            
            try:                
                trigger = ctx.triggered[0]['prop_id'].split('.')[0] if ctx.triggered else None
            except Exception as e:
                trigger = "no trigger"

            try:                
                selected_source = ctx.inputs.get('selected-source.data', None)
            except Exception as e:
                selected_source = None

            try:                
                selected_ids = ctx.inputs.get('selected-ids.data', None)
            except Exception as e:
                selected_ids = None

            result = func(*args, **kwargs)
            elapsed = time.perf_counter() - start
            #logging.debug(f"{elapsed:.3f}s used by callback {name or func.__name__}({trigger}, selected-source={selected_source}, # selected-ids={len(selected_ids) if selected_ids else 0}).")
            return result
        return wrapper
    return decorator


# Prepare hover text for selected
def make_hover_text(df_):
    # Use the columns currently visible in the data table (from dcc.Store)
    # Fallback to all columns if not available
    try:
        from dash import callback_context
        ctx = callback_context
        # Try to get visible columns from dcc.Store (if available)
        visible_columns = ['id'] #ctx.states.get('visible-columns.data', None)
    except Exception:
        visible_columns = None
    if not visible_columns:
        visible_columns = [col for col in df_.columns if col != 'Selected']

    # Compose hover text for each row
    hover_text = []
    for _, row in df_.iterrows():
        lines = [f"{col}: {row[col]}" for col in visible_columns if col in df_.columns]
        hover_text.append("<br>".join(lines))
    return hover

In [ ]:
####################

In [ ]:
## generic figures

def get_marker_dict(group, df, aesthetics_group, legend=True, continuous=False, unselected=False, mapplot=False):
    """
    Build a Plotly marker dict for different cases (categorical, continuous, unselected).
    """
    point_color = aesthetics_group['color']
    point_size = aesthetics_group['size']
    point_opacity = aesthetics_group['opacity']
    point_symbol = aesthetics_group['symbol'] if not mapplot else aesthetics_group['symbol_map']

    # Unselected case
    if unselected:
        marker = dict(
            marker=dict(
                size=point_size['unselected'],
                color=point_color['unselected'],
                # symbol=point_symbol['unselected'], does not exist
                opacity=point_opacity['unselected']
                )
            )

    # Continuous color scale
    elif continuous:
        marker = dict(
            size=point_size.get(group, point_size['default']),
            opacity=point_opacity.get(group, point_opacity['default']),
            symbol=point_symbol.get(group, point_symbol['default']),
            color=df[group],  # numeric values needed for continuous coloring
            colorscale=point_color['colorscale'],
            colorbar=dict(title=group) if legend else None,
            showscale=legend
        )

    # Categorical color
    else:
        marker = dict(
            size=point_size.get(group, point_size['default']),
            opacity=point_opacity.get(group, point_opacity['default']),
            symbol=point_symbol.get(group, point_symbol['default']),
            color=point_color[group] ## color is always set
        )

    logging.info(f"{'continuous'if continuous else ('unselected' if unselected else 'selected')}: {marker}")
    return marker


## 2D scatter plot
def generate_fig_Scatter(x_col, y_col, group, aesthetics_group, legend=True, xlab=True, ylab=True, df=df):
    traces = []

    if group == 'none':
        ## no grouping at all
        traces.append(go.Scatter(
            x=df[x_col],
            y=df[y_col],
            mode='markers',
            marker=get_marker_dict(group, df, aesthetics_group),
            unselected=get_marker_dict(group, df, aesthetics_group, unselected=True),
            name=str(group),
            customdata=df['id'],
            text=group,   
            showlegend=legend
        ))
    elif df[group].dtype.kind in 'fi':
        # Use color array for continuous
        traces.append(go.Scatter(
            x=df[x_col],
            y=df[y_col],
            mode='markers',
            marker=get_marker_dict(group, df, aesthetics_group, legend=legend, continuous=True),
            unselected=get_marker_dict(group, df, aesthetics_group, unselected=True),
            name=group,
            customdata=df['id'],
            text=df[group],
            showlegend=False  # Only one colorbar for continuous
        ))
    else:
        # Categorical: single color per group
        for g, group_df in df.groupby(group, sort=False):
            traces.append(go.Scatter(
                x=group_df[x_col],
                y=group_df[y_col],
                mode='markers',
                marker=get_marker_dict(g, df, aesthetics_group),
                unselected=get_marker_dict(g, df, aesthetics_group, unselected=True),
                name=str(g),
                customdata=group_df['id'],
                text=group_df[group],   
                showlegend=legend
            ))

    fig = go.Figure(traces)
    fig.update_layout(
        xaxis_title=x_col if xlab else '',  
        yaxis_title=y_col if ylab else '',
    )
    return fig

## 2d scatter plot go.Scattergl: should be faster for large tables, but does not produce a vectorized image (currently not used)
def generate_pca_fig_Scattergl(x_col, y_col, group, aesthetics_group, legend=True, xlab=True, ylab=True, df=df):
    traces = []

    if group == 'none':
        ## no grouping at all      
        traces.append(go.Scattergl(
            x=df[x_col],
            y=df[y_col],
            mode='markers',
            marker=get_marker_dict(group, df, aesthetics_group),
            unselected=get_marker_dict(group, df, aesthetics_group, unselected=True),
            name=str(group),
            customdata=df['id'],
            text=df[group],
            showlegend=True
        ))
    elif df[group].dtype.kind in 'fi':
        assert(isinstance(color_map, dict) & ('colorscale' in color_map)), "For continuous group, color_map must be a dict with keys: colorscale"
        # Use color array for continuous
        traces.append(go.Scattergl(
            x=df[x_col],
            y=df[y_col],
            mode='markers',
            marker=get_marker_dict(group, df, aesthetics_group, legend=legend, continuous=True),
            unselected=get_marker_dict(group, df, aesthetics_group, unselected=True),
            name=group,
            customdata=df['id'],
            text=df[group],
            showlegend=False  # Only one colorbar for continuous
        ))
    else:
        # Categorical: single color per group
        for g, group_df in df.groupby(group, sort=False):
            if group_df.empty:
                continue
            traces.append(go.Scattergl(
                x=group_df[x_col],
                y=group_df[y_col],
                mode='markers',
                marker=get_marker_dict(g, df, aesthetics_group),
                unselected=get_marker_dict(g, df, aesthetics_group, unselected=True),
                name=str(g),
                customdata=group_df['id'],
                text=group_df[group],
                showlegend=True
            ))
        fig = go.Figure(traces)
    return fig


## 3d scatter plot
def generate_fig_Scatter3d(x_col, y_col, z_col, group, aesthetics_group, legend=True, xlab=True, ylab=True, zlab=True, df=df, selected_ids=[]):
    traces = []

    if group == 'none':
        ## no grouping at all
        traces.append(go.Scatter3d(
            x=df[x_col],
            y=df[y_col],
            z=df[z_col],
            mode='markers',
            #marker=get_marker_dict(group, df, aesthetics_group),
            #unselected=get_marker_dict(group, df, aesthetics_group, unselected=True), # it does nto exist
            name=str(group),
            customdata=df['id'],
            text=group,   
            showlegend=legend
        ))

    else:        
        ## split df into selected and unselected
        df_selected, df_unselected = get_selected_df_both(selected_ids, df)
    
        if len(selected_ids) < len(df):
            # Unselected points (gray, no legend)        
            traces.append(go.Scatter3d(
                x=df_unselected[x_col],
                y=df_unselected[y_col],
                z=df_unselected[z_col],
                mode='markers',
                marker=get_marker_dict(group, df, aesthetics_group, unselected=True)["marker"],
                name='Unselected',
                customdata=df_unselected['id'],
                showlegend=False,
                hoverinfo='skip'
            ))
        
        if df[group].dtype.kind in 'fi':
            # Use color array for continuous
            traces.append(go.Scatter3d(
                x=df_selected[x_col],
                y=df_selected[y_col],
                z=df_selected[z_col],
                mode='markers',
                marker=get_marker_dict(group, df, aesthetics_group, legend=legend, continuous=True),
                name=group,
                customdata=df_selected['id'],
                text=df_selected[group],
                showlegend=False  # Only one colorbar for continuous
            ))
        else:
            # Categorical: single color per group
            for g, group_df in df_selected.groupby(group, sort=False):
                if group_df.empty:
                    continue
                traces.append(go.Scatter3d(
                    x=group_df[x_col],
                    y=group_df[y_col],
                    z=group_df[z_col],
                    mode='markers',
                    marker=get_marker_dict(g, df, aesthetics_group),
                    name=str(g),
                    customdata=group_df['id'],
                    text=group_df[group],
                    showlegend=legend
                ))

    fig = go.Figure(traces)
    fig.update_layout(
        scene=dict(
            xaxis_title=x_col if xlab else '',
            yaxis_title=y_col if ylab else '',
            zaxis_title=z_col if zlab else ''
        ),
        legend=dict(title=group)
    )
    return fig

def generate_pca_fig_Scatter3d(df_selected, df_unselected, x_col, y_col, z_col, group, point_size, color_map):
    traces = []

    # Unselected points (gray, no legend)
    if not df_unselected.empty:
        traces.append(go.Scatter3d(
            x=df_unselected[x_col],
            y=df_unselected[y_col],
            z=df_unselected[z_col],
            mode='markers',
            marker=dict(size=point_size, color='lightgray', opacity=0.3),
            name='Unselected',
            customdata=df_unselected['id'],
            showlegend=False,
            hoverinfo='skip'
        ))

    # Selected points (colored by group, legend per group)
    if not df_selected.empty:
        if df_selected[group].dtype.kind in 'fi':
            # Use color array for continuous
            traces.append(go.Scatter3d(
                x=df_selected[x_col],
                y=df_selected[y_col],
                z=df_selected[z_col],
                mode='markers',
                marker=dict(
                    size=point_size,
                    color=df_selected[group],
                    colorscale='Viridis',
                    opacity=0.9,
                    colorbar=dict(title=group)
                ),
                name=group,
                customdata=df_selected['id'],
                text=df_selected[group],
                showlegend=False  # Only one colorbar for continuous
            ))
        else:
            # Categorical: single color per group
            for g, group_df in df_selected.groupby(group, sort=False):
                if group_df.empty:
                    continue
                traces.append(go.Scatter3d(
                    x=group_df[x_col],
                    y=group_df[y_col],
                    z=group_df[z_col],
                    mode='markers',
                    marker=dict(size=point_size, color=color_map[g], opacity=0.9),
                    name=str(g),
                    customdata=group_df['id'],
                    text=group_df[group],
                    showlegend=True
                ))

    fig = go.Figure(traces)
    return fig

In [ ]:

## PCA figure

@lru_cache(maxsize=32)
def generate_pca_fig(aesthetics_store_tuple, x_col, y_col, z_col, group, plot_type, selected_ids=None):
    aesthetics_group = tuple_to_dict_of_dicts(aesthetics_store_tuple)

    logging.debug(f"generate_pca_fig for group {group} with aesthetics_group {aesthetics_group}")

    if plot_type == '2D':
        fig = generate_fig_Scatter(x_col, y_col, group, aesthetics_group, legend=True, df=df)
    else:
        selected_ids = tuple(selected_ids) if selected_ids else ()
        fig = generate_fig_Scatter3d(x_col, y_col, z_col, group, aesthetics_group, legend=True, df=df, selected_ids=selected_ids)

    fig.update_layout(
        template='plotly_white',
        clickmode='event+select',
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        title="",
        margin={'l': 0, 'r': 0, 't': 0, 'b': 0},
        dragmode='lasso',
        legend_title=group
    )

    # Create traces: group_id → trace index in fig.data
    traces = {trace.name: i for i, trace in enumerate(fig.data)}

    return fig, traces

In [ ]:
## Map figure

## Optimized version using Scattermap (slow, but works with selection)
def generate_map_fig_Scattermap(group, aesthetics_group, legend=True):
    traces = []

    if group == 'none':
        # no gouping
        traces.append(go.Scattermap(
            lat=df[ANNOTATION_LAT],
            lon=df[ANNOTATION_LONG],
            mode='markers',
            marker=get_marker_dict(group, df, aesthetics_group, mapplot=True),
            unselected=get_marker_dict(group, df, aesthetics_group, unselected=True, mapplot=True),
            name=group,
            customdata=df['id'],
            text=group,
            showlegend=legend,
        ))
    if df[group].dtype.kind in 'fi':
        # Continuous
        traces.append(go.Scattermap(
            lat=df[ANNOTATION_LAT],
            lon=df[ANNOTATION_LONG],
            mode='markers',
            marker=get_marker_dict(group, df, aesthetics_group, legend=legend, continuous=True, mapplot=True),
            unselected=get_marker_dict(group, df, aesthetics_group, unselected=True, mapplot=True),
            name=group,
            customdata=df['id'],
            text=df[group],
            showlegend=legend,  
        ))
    else:
        # Categorical: single color per group
        for g, group_df in df.groupby(group, sort=False):
            if group_df.empty:
                continue
            traces.append(go.Scattermap(
                lat=group_df[ANNOTATION_LAT],
                lon=group_df[ANNOTATION_LONG],
                mode='markers',
                marker=get_marker_dict(g, df, aesthetics_group, mapplot=True),
                unselected=get_marker_dict(g, df, aesthetics_group, unselected=True, mapplot=True),
                name=str(g),
                customdata=group_df['id'],
                text=group_df[group],
                showlegend=legend,
            ))

    fig = go.Figure(traces)
    return fig



@lru_cache(maxsize=32)
def generate_map_fig(aesthetics_store_tuple, group):
    aesthetics_group = tuple_to_dict_of_dicts(aesthetics_store_tuple)
    
    fig = generate_map_fig_Scattermap(group, aesthetics_group, legend=args.show_all_legends)

    fig.update_layout(
        mapbox_style='open-street-map', 
        margin={'l': 0, 'r': 0, 't': 0, 'b': 0},
        title='',
        dragmode='lasso',
        showlegend=args.show_all_legends
    )

    # Create traces: group_id → trace index in fig.data
    traces = {trace.name: i for i, trace in enumerate(fig.data)}

    return fig, traces



In [ ]:
## Time figure
def generate_time_histogram_simple(var_continuous, nbins):
    fig = px.histogram(df, x=var_continuous, nbins=nbins, title="", color_discrete_sequence=['#1F77B4'])
    fig.update_layout(yaxis_title='Count')
    return fig


def generate_time_histogram(df_selected, var_continuous, nbins=args.time_hist_nbins):
    # All points (gray)
    trace_all = go.Histogram(
        x=df[var_continuous],
        nbinsx=nbins,
        marker_color='lightgray',
        opacity=0.6,
        name='All'
    )
    traces = [trace_all]

    # Selected points (color)
    if not df_selected.empty:
        trace_selected = go.Histogram(
            x=df_selected[var_continuous],
            nbinsx=nbins,
            marker_color='#1F77B4',
            opacity=0.9,
            name='Selected'
        )
        traces.append(trace_selected)

    fig = go.Figure(traces)

    fig.update_layout(
        barmode='overlay',
        yaxis_title='Count',
        showlegend=True
    )
    return fig


def get_time_range(df_selected):
    if not df_selected.empty:
        series = df_selected[ANNOTATION_TIME].dropna()
    else:
        series = df[ANNOTATION_TIME].dropna()

    if not series.empty: ## empty or NaN
        start = int(series.min())
        end = int(series.max())
    else:
        start = 0
        end = 1

    return start, end


## Histogram figure
@lru_cache(maxsize=32)
def generate_time_fig(selected_ids, aesthetics_store_tuple, var_continuous, group, plot_type, time_slider=False):
    assert(ANNOTATION_TIME is not None)

    df_selected = get_selected_df(selected_ids)

    if plot_type == 0: # 'scatter':
        aesthetics_group = tuple_to_dict_of_dicts(aesthetics_store_tuple)
        #assert('color' in aesthetics_group), "Color aesthetics must be provided for scatter plot."
        #assert(set(df_selected[group].unique()) <= set(aesthetics_group['color'].keys()) and ), "Color map keys do not match selected group values."
        logging.debug(f"generate_time_fig for group {group} with aesthetics_group {aesthetics_group}")

        df_plot = df.copy()
        jiggle = 0.3
        df_plot['jiggle'] = np.random.uniform(-jiggle, jiggle, size=len(df_plot))
        fig = generate_fig_Scatter(var_continuous, 'jiggle', group, aesthetics_group, 
                                   legend=args.show_all_legends, ylab=False, df=df_plot)

    elif plot_type == 1: #'histogram':
        df_selected = get_selected_df(selected_ids)
        fig = generate_time_histogram(df_selected, var_continuous, args.time_hist_nbins)

    else: # plot_type == 'histogram_simple':
        df_selected = get_selected_df(selected_ids)
        fig = generate_time_histogram_simple(var_continuous, args.time_hist_nbins)

    fig.update_layout(
        template='plotly_white',
        clickmode='event+select',
        margin={'l': 0, 'r': 0, 't': 0, 'b': 40},
        paper_bgcolor='rgba(0,0,0,0)')

    if time_slider:
        ## add range slider
        start, end = get_time_range(df_selected)

        fig.update_layout(
            template='plotly_white',
            #showlegend=False,
            yaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
            xaxis=dict(
                title=f"{var_continuous} from {start:,} to {end:,}",
                rangeslider=dict(
                    visible=True,
                    thickness=0.10
                ),
                type="linear"
            ),
            uirevision='time-plot'
        )
    
    # Create traces: group_id → trace index in fig.data
    traces = {trace.name: i for i, trace in enumerate(fig.data)}

    return fig, traces

In [ ]:
## aesthetics store functions
def get_color_map(group):
    logging.debug(f"Generating color map for group '{group}' ...")

    # Continuous → colorscale ONLY
    if df[group].dtype.kind in 'fi':
        return {'colorscale': args.color_schema_continuous}

    # Categorical
    unique_values = df[group].dropna().unique()
    assert len(unique_values) > 0, f"No unique values found in group column '{group}'."

    color_map = {}
    px_colors = px.colors.qualitative.Plotly
    for i, val in enumerate(unique_values):
        color_map[val] = px_colors[i % len(px_colors)]

    logging.debug(f"Color map: {color_map}")
    return color_map


def get_init_aesthetics_of_group(group):
    if args.aesthetics_file:
        data = read_dict_of_dicts_from_json(args.aesthetics_file)
    elif group:
            data={
                'color': {
                     'default': args.point_color, 
                     'unselected': args.point_color_unselected, 
                     **get_color_map(group)
                     }, 
                'size': {
                     'default': args.point_size, 
                     'unselected': args.point_size_unselected
                     },
                'opacity': {
                     'default': args.point_opacity, 
                     'unselected': args.point_opacity_unselected
                     },
                'symbol': {
                     'default': args.point_symbol, 
                     'unselected': args.point_symbol_unselected
                     },
                'symbol_map': {
                     'default': args.point_symbol if args.point_symbol in maplot_point_symbols else 'circle', 
                     'unselected': args.point_symbol_unselected if args.point_symbol_unselected in maplot_point_symbols else 'circle' ## mapplot has not all symbols as the plotly
                     },
            }
    else: 
         data = {}
         
    logging.debug(f"Initial aesthetics store: {data}")
    return data

In [ ]:
####################

In [ ]:
## PCA panel
def get_selectors(plot_type='2D'):
    logging.debug(f"   get_selectors() ... ")
    dropdowns = []

    # X-axis
    dropdowns.append(
        html.Div([
            html.Div("X-axis:", style={"marginRight": "5px"}),
            dcc.Dropdown(id='x-axis-selector', 
                         options=PCS, 
                         value=init_x,
                         clearable=False, 
                         style={'width': '120px'}),
        ], id='x-axis-div', style={'display': 'flex', 'alignItems': 'center', 'marginRight': '20px'})
    )

    # Y-axis
    dropdowns.append(
        html.Div([
            html.Div("Y-axis:", style={"marginRight": "5px"}),
            dcc.Dropdown(id='y-axis-selector', 
                         options=PCS, 
                         value=init_y,
                         clearable=False, 
                         style={'width': '120px'}),
        ], id='y-axis-div', style={'display': 'flex', 'alignItems': 'center', 'marginRight': '20px'})
    )

    # Z-axis
    dropdowns.append(
        html.Div([
            html.Div("Z-axis:", style={"marginRight": "5px"}),
            dcc.Dropdown(id='z-axis-selector', 
                         options=PCS, 
                         value=init_z,
                         clearable=False, 
                         style={'width': '120px'}),
        ], id='z-axis-div', style={'display': 'flex', 'alignItems': 'center', 'marginRight': '20px'})
    )
    
    # Grouping (optional)
    if dropdown_group_list:
        dropdowns.append(
            html.Div([
                html.Div("Coloring:", style={"marginRight": "5px"}),
                dcc.Dropdown(id='group-selector', 
                            options=[{'label': abbr, 'value': abbr} for abbr in dropdown_group_list],
                            value=init_group,
                            clearable=False, 
                            style={'width': '200px'}),
            ], id='coloring-div', 
            style={'display': 'flex' if (len(dropdown_group_list) > 1) else 'none',  # hide if single option 
                   'alignItems': 'center', 
                   'marginRight': '20px'})
        )
        dropdowns.append(html.Button("Aesthetics", id="show-table-btn"))


    selectors = html.Div(dropdowns, style={'display': 'flex', 'flexWrap': 'wrap', 'marginBottom': '15px'})
    logging.debug(f"   get_selectors() ... done.")
    return selectors


def get_PCA_panel():
    return html.Div(
        id='pca-panel',
        style={'flex': 1, 'display': 'flex', 'flexDirection': 'column', 'width': '100%', 'height': '100%'},
        children=[
            html.Div(
                style={
                    'display': 'flex',
                    'alignItems': 'center',
                    'gap': '20px',        # space between controls
                    'marginBottom': '5px'
                },
                children=[
                    dcc.RadioItems(
                        id='pca-plot-type-switch',
                        options=[{'label': '2D', 'value': '2D'}, {'label': '3D', 'value': '3D'}],
                        value='2D',
                        inline=True,
                        style={'marginRight': '10px'}
                    ),
                    dcc.Checklist(
                        id='show-legend',
                        options=[{'label': 'Show legend', 'value': 'show'}],
                        value=['show'],  # default: legend visible
                        inline= len(dropdown_group_list) > 1,
                        style={'marginBottom': '5px'}
                    )
                ]),
            html.Div(
                id='selectors-container', 
                children=get_selectors()
                ),

            dcc.Graph(
                id='pca-plot',
                figure=init_pca_fig, 
                style={'flex': 1, 'width': '100%', 'height': '100%'}
            )
        ]
    )



In [ ]:
## Map panel
def get_map_panel():
    return html.Div(
        style={'flex': 1, 'display': 'flex', 'flexDirection': 'column', 'width': '100%', 'height': '100%'},
        children=[
            html.Div(
                id='selection-count',
                style={'padding': '10px', 'fontWeight': 'bold'},
                children=f"Showing all {len(df)} points"
            ),
            html.Div(id="display-text"),
            html.Div(
                dcc.Graph(
                    id='map-plot', 
                    figure=init_map_fig,
                    style={'height': '100%', 'width': '100%'}),
                style={'flex': 1, 'display': 'flex'}
            )
        ]
    )

In [ ]:
## Time panel
def get_range_slider(df_selected=None):
    startTimeTot, endTimeTot = get_time_range(df)

    if df_selected is not None:
        startTimeSel, endTimeSel = get_time_range(df_selected)
    else:
        startTimeSel, endTimeSel = startTimeTot, endTimeTot

    fig = dcc.RangeSlider(
        id='time-range',
        min=startTimeTot,
        max=endTimeTot,
        value=[startTimeSel, endTimeSel],
        #marks={i: str(date)[:10] for i, date in enumerate(df[ANNOTATION_TIME].unique()[::50])},
        tooltip={"placement": "bottom", "always_visible": False},
        allowCross=False,
    )

    return fig



def get_time_panel():
    if ANNOTATION_TIME is None:
        return html.Div()
    
    option_childern = [
        dcc.RadioItems(
                id='time-plot-type-switch',
                options=[{'label': 'Scatter', 'value': 0}, {'label': 'Histogram', 'value': 1}, {'label': 'Histogram simple', 'value': 2}],
                value=0,
                inline=True,
                style={'marginRight': '20px'}
            ),
            #get_range_slider(),
    ]

    if dropdown_list_continuous:
        option_childern.append(
            html.Div([
                html.Div(
                    "Variable:", 
                    style={"marginRight": "5px"}
                ),

                dcc.Dropdown(
                    id='time-selector', 
                    options=[{'label': abbr, 'value': abbr} for abbr in dropdown_list_continuous],
                    value=init_continuous,
                    clearable=False, 
                    style={'width': '200px'}
                ),
            ], id='time-div', style={'display': 'flex', 'alignItems': 'center', 'marginRight': '20px'})
        )
         
    # Time line
    return html.Div(
        style={'flex': 1, 'display': 'flex', 'flexDirection': 'column', 'width': '100%', 'height': '100%'},
        children=[
            html.Div(
                option_childern, 
                style={'display': 'flex', 'alignItems': 'center', 'gap': '12px', 'marginBottom': '15px'}
            ),

            dcc.Graph(
                id='time-plot', 
                figure=init_time_fig,
                style={'height': '100%', 'width': '100%'}
            ),
        ])

In [ ]:
## Data table panel
## here we mimic a checkbox, thus no rerendering has to be done
def get_data_table_column_defs(selected_columns):
    if selected_columns is None:
        return annotation_desc_ext['Abbreviation'].head(1).tolist()

    return ([{"headerName": " ", "field": "Selected", "editable": True, "cellEditor": "agCheckboxCellEditor", "cellRenderer": "agCheckboxCellRenderer", "width": 60, "suppressMenu": True, "filter": "agTextColumnFilter"}] +
                    [{"headerName": col, "field": col, "sortable": True, "resizable": True} for col in selected_columns])


def get_data_table_AgGrid():
    if "Selected" not in df.columns:
        df["Selected"] = True

    # Build the AgGrid component
    tab = AgGrid(
        id="data-table",
        rowData=df.to_dict("records"),
        columnDefs=get_data_table_column_defs(annotation_desc_ext['Abbreviation'].head(10).tolist()),
        dashGridOptions={
            "suppressRowClickSelection": True,
            "pagination": True,
            "paginationAutoPageSize": True,
            "animateRows": True,
            "defaultColDef": {
                "sortable": True,
                "filter": True,
                "resizable": True,
            },
        },
        style={"height": "100%", "width": "100%"},
        className="ag-theme-alpine",
    )

    return tab

def get_data_table_panel():
    return html.Div(
        style={'flex': 1, 'display': 'flex', 'flexDirection': 'column', 'width': '100%', 'height': '100%'},
        children=[
            html.Div(
                get_data_table_AgGrid(),
                style={'flex': '1 1 auto', 'width': '100%', 'height': 'calc(100% - 80px)', 'minHeight': 0}
            ),
            html.Div(
                dcc.Textarea(
                    id='texarea-expression', 
                    placeholder='Filter samples using pandas.query()',
                    style={'width': '100%', 'height': '80px', "boxSizing": "border-box", 'resize': 'none', 'padding': '8px',  }
                ),
                style={'flex': '0 0 80px', 'width': '100%'}
            )
        ]
    )

In [ ]:
####################

In [ ]:
## PCA tab

def get_pca_tab():
    left_children = [
        html.Div(
            get_PCA_panel(),
            id='pca_panel',
            style={'flex': '1 1 80%', 'minHeight': '100px'}
        ),
    ]

    if ANNOTATION_TIME:
        left_children.extend([
            html.Div(
                id='vertical-divider-left',
                style={
                    'height': '6px',
                    'cursor': 'row-resize',
                    'backgroundColor': '#ddd',
                    'zIndex': 10,
                    'marginTop': '10px',
                    'marginBottom': '10px'
                }
            ),
            html.Div(
                get_time_panel(),
                id='time-plot-panel',
                style={'flex': '1 1 20%', 'minHeight': '100px'}
            ),
        ])

    right_children = []
    if ANNOTATION_LAT:
        right_children.extend([
            html.Div(
                get_map_panel(),
                id='map_panel',
                style={'flex': '1 1 50%', 'minHeight': '100px'}
            ),
            html.Div(
                id='vertical-divider-right',
                style={
                    'height': '6px',
                    'cursor': 'row-resize',
                    'backgroundColor': '#ddd',
                    'zIndex': 10,
                    'marginTop': '10px',
                    'marginBottom': '10px'
                }
            )
        ])

    if annotation_desc is not None:
        right_children.append(
            html.Div(
                get_data_table_panel(),
                id='data-table-panel',
                style={'flex': '1 1 50%', 'minHeight': '100px'}
            )
        )


    ## put left and right children together
    split_container_children = [
        html.Div(
            left_children,
            id='left-panel',
            style={
                'flex': '1 1 100%' if not right_children else '0 0 50%',
                'padding': '10px',
                'minWidth': '200px',
                'display': 'flex',
                'flexDirection': 'column',
                'overflow': 'hidden'
            },
        )
    ]

    if right_children:
        split_container_children.extend([
            html.Div(
                id='divider',
                style={
                    'width': '6px',
                    'cursor': 'col-resize',
                    'backgroundColor': '#ddd',
                    'zIndex': 10,
                    'marginTop': '10px',
                    'marginBottom': '10px'
                }
            ),
            html.Div(
                right_children,
                id='right-panel',
                style={
                    'flex': '1 1 auto',
                    'padding': '10px',
                    'minWidth': '200px',
                    'display': 'flex',
                    'flexDirection': 'column',
                    'overflow': 'hidden'
                }
            ),
        ])

    tab = html.Div(
        id='pca_tab_content',
        children=[
            html.Div(
                id='split-container',
                children=split_container_children,
                style={
                    'display': 'flex',
                    'flexDirection': 'row',
                    'height': 'calc(100vh - 80px)',
                    'overflow': 'hidden'
                }
            ),
            # store to trigger clientside script initialization
            dcc.Store(id='init-trigger', data=0),
        ]
    )

    return tab

In [ ]:
## Annotation tab
from flask import app

def get_annotation_tab_AgGrid():
    logging.debug("get_annotation_tab_AgGrid() ... ")

    if annotation_desc is None:
        return None

    # Add a 'Selected' checkbox column
    if "Selected" not in annotation_desc_ext.columns:
        annotation_desc_ext["Selected"] = False

    # Preselect first 10 rows
    annotation_desc_ext.loc[annotation_desc_ext.index[:10], "Selected"] = True

    # Column definitions
    column_defs = [{"headerName": "", "field": "Selected", "editable": True, "cellEditor": "agCheckboxCellEditor", "cellRenderer": "agCheckboxCellRenderer", "width": 60, "suppressMenu": True}]

    # Add remaining columns with width rules
    for i, col in enumerate([c for c in annotation_desc_ext.columns if c != "Selected"]):
        if i == 1:  # Column 2 → fill remaining space
            column_defs.append({"headerName": col, "field": col, "flex": 1, "sortable": True, "resizable": True})
        else:       # Columns 1,3,4,5 → auto width / adapt to text
            column_defs.append({"headerName": col, "field": col, "sortable": True, "resizable": True})


    tab = html.Div(
        id="annotation_tab_content",
        children=[
            AgGrid(
                id="annotation-table",
                rowData=annotation_desc_ext.to_dict("records"),
                columnDefs=column_defs,
                dashGridOptions={
                    "suppressRowClickSelection": True,
                    "animateRows": True,
                    "defaultColDef": {
                        "resizable": True,
                        "wrapText": True,
                        "autoHeight": True,
                    },
                },

                className="ag-theme-alpine",
                style={"height": "calc(100vh - 80px)", "width": "100%"},
            )
        ],
        style={"display": "flex", "width": "100%", "height": "calc(100vh - 80px)", "flexDirection": "column"},
    )

    logging.debug("get_annotation_tab_AgGrid() ... done.")
    return tab

In [ ]:
## Eigenvalues tab
def get_eigenvalues_tab():
    logging.debug(f"get_eigenvalues_tab() ... ")
    if args.eigenval is None:
        return None

    tab = html.Div([
        html.Div([
            dcc.Graph(
                id='eigenvals',
                figure={
                    'data': [{
                        'x': eigenval["dimension"][:args.nb_eigenvalues],
                        'y': 100 * eigenval['eigenvalue'][:args.nb_eigenvalues],
                        'type': 'bar',
                        'name': 'Eigenvalues'
                    }],
                    'layout': {
                        'title': {'text': 'Eigenvalues'},
                        'xaxis': {'title': {'text': 'Dimension'}},
                        'yaxis': {'title': {'text': '% explained variance'}},
                        'autosize': True,
                        'height': None,
                    }
                },
                style={'width': '50%', 'height': '100%', 'display': 'flex', 'verticalAlign': 'top'}
            ),
            dcc.Graph(
                id='eigenvals_cumulative',
                figure={
                    'data': [{
                        'x': eigenval["dimension"][:args.nb_eigenvalues],
                        'y': 100 * eigenval['cumulative'][:args.nb_eigenvalues],
                        'type': 'bar',
                        'name': 'Cumulative'
                    }],
                    'layout': {
                        'title': {'text': 'Cumulative Eigenvalues'},
                        'xaxis': {'title': {'text': 'Dimension'}},
                        'yaxis': {'title': {'text': '% cumulative explained variance'}},
                        'autosize': True,
                        'height': None,
                    }
                },
                style={'width': '50%', 'height': '100%', 'display': 'flex', 'verticalAlign': 'top'}
            ),
        ], style={
            'width': '100%',
            'height': '100%',
            'display': 'flex',
            'flexDirection': 'row',
            'justifyContent': 'space-between',
            'alignItems': 'stretch'
        }),
    ], 
    id='eigenvalues_tab_content', 
    style={'height': 'calc(100vh - 80px)', 'width': '100%'})

    logging.debug(f"get_eigenvalues_tab() ... done")
    return tab




In [ ]:
## Statistics tab
def get_statistics_tab():
    logging.debug(f"get_statistics_tab() ... ")
    if df_imiss is None and df_lmiss is None and df_frq is None:
        return None  # Don't render the tab at all

    graphs = []

    ## imiss
    if df_imiss is not None:
        fig_imiss = px.histogram(
            df_imiss, x='F_MISS', nbins=50, title='Sample missing rate',
            labels={'F_MISS': 'Missing rate', 'count': 'Count'},
            color_discrete_sequence=['#1F77B4']
        )
        fig_imiss.update_layout(
            template='plotly_white',
            xaxis=dict(range=[0, 1]), 
            autosize=True
        )
        graphs.append(
            dcc.Graph(id='imiss', figure=fig_imiss, style={
                'flex': 1, 'height': '100%', 'width': '100%',
                'display': 'inline-block', 'verticalAlign': 'top'
            })
        )

    ## lmiss
    if df_lmiss is not None:
        fig_lmiss = px.histogram(
            df_lmiss, x='F_MISS', nbins=50, title='SNP missing rate',
            labels={'F_MISS': 'Missing rate', 'count': 'Count'},
            color_discrete_sequence=['#1F77B4']
        )
        fig_lmiss.update_layout(
            template='plotly_white',
            xaxis=dict(range=[0, 1]), 
            autosize=True
        )
        graphs.append(
            dcc.Graph(id='lmiss', figure=fig_lmiss, style={
                'flex': 1, 'height': '100%', 'width': '100%',
                'display': 'flex', 'verticalAlign': 'top'
            })
        )

    ## freq
    if df_frq is not None:
        fig_frq = px.histogram(
            df_frq, x='MAF', nbins=500, title='Minor Allele Frequency',
            labels={'MAF': 'Allele frequency', 'count': 'Count'},
            color_discrete_sequence=['#1F77B4']
        )
        fig_frq.update_layout(
            template='plotly_white',
            xaxis=dict(range=[0, 0.5]), 
            autosize=True
        )
        graphs.append(
            dcc.Graph(id='freq', figure=fig_frq, style={
                'flex': 1, 'height': '100%', 'width': '100%',
                'display': 'flex', 'verticalAlign': 'top'
            })
        )

    tab = html.Div([
        html.Div(graphs, style={
            'width': '100%',
            'height': '100%',
            'display': 'flex',
            'flexDirection': 'row',
            'justifyContent': 'space-between',
            'alignItems': 'stretch'
        }),
    ], id='statistics_tab_content', 
    style={'height': 'calc(100vh - 80px)', 'width': '100%'})

    logging.debug(f"get_statistics_tab() ... done.")
    return tab




In [ ]:
## Help tab

## remove color encoding from argparse help
ANSI_ESCAPE_RE = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])')

def strip_ansi(text):
    return ANSI_ESCAPE_RE.sub('', text)

def get_help_tab():
    logging.debug(f"get_help_tab() ... ")


    tab = html.Div(
        id='help_tab_content',
        children=[
            html.Pre(
                strip_ansi(parser.format_help()),
                style={
                    'height': '100%',
                    "minHeight": 0,
                    'overflowY': 'auto',
                    'whiteSpace': 'pre-wrap',
                    #'wordWrap': 'break-word',
                    'backgroundColor': '#f8f9fa',
                    'padding': '10px',
                    'border': '1px solid #ddd',
                    'fontFamily': 'monospace',
                    'fontSize': '14px',
                    'margin': 0,
                    'flex': '1 1 auto',
                }
            )
        ],
        style={'display': 'flex', 
            'width': '100%', 
            'height': 'calc(100vh - 80px)', 
            "flexDirection": "column"}
    )
    
    return tab


In [ ]:
####################

In [ ]:
## generate init objects
logging.debug(init_continuous)
init_aesthetics = get_init_aesthetics_of_group(init_group)

init_pca_fig, init_pca_trace = generate_pca_fig(dict_of_dicts_to_tuple(init_aesthetics), init_x, init_y, init_z, init_group, '2D', tuple(init_selected_ids))

if ANNOTATION_LAT:
    init_map_fig, init_map_trace = generate_map_fig(dict_of_dicts_to_tuple(init_aesthetics), init_group)
else:
    init_map_fig, init_map_trace = None, None

if ANNOTATION_TIME:
    init_time_fig, init_time_trace = generate_time_fig(tuple(init_selected_ids), dict_of_dicts_to_tuple(init_aesthetics), init_continuous, init_group, args.time_plot_type)
else:
    init_time_fig, init_time_trace = None, None

In [ ]:
## dash layout

## get tabs dynamically
def labeled_tabs(visible_tab_ids):
    tab_labels = {
        'pca_tab': 'PCA',
        'annotation_tab': 'Annotation',
        'eigenvalues_tab': 'Eigenvalues',
        'statistics_tab': 'Statistics',
        'help_tab': 'Help'
    }

    return html.Div(
        style={'display': 'flex', 'alignItems': 'center', 'width': '100%'},
        children=[
            # Left title
            html.Img(
                src='/assets/dbc_logo_400x400.jpg',
                style={'height': '70px', 'width': '70px', 'marginRight': '10px', 'marginTop': '5px', 'marginLeft': '5px'}
            ),
            html.Div(
                "interactivePCA",
                style={'fontWeight': 'bold', 'marginRight': '20px', 'marginLeft': '0px', 'fontSize': '30px', 'whiteSpace': 'nowrap'}
            ),

            # Tabs
            html.Div(
                style={'flex': 1},  # make tabs container take remaining space
                children=dcc.Tabs(
                    id='tabs',
                    value=visible_tab_ids[0] if visible_tab_ids else None,
                    style={'width': '100%'},  # stretch container
                    children=[
                        dcc.Tab(
                            label=tab_labels[tab_id],
                            value=tab_id,
                            style={'flex': 1, 'textAlign': 'center'},
                            selected_style={'flex': 1, 'textAlign': 'center', 'fontWeight': 'bold'}
                        )
                        for tab_id in visible_tab_ids
                    ]
                )
            )
        ]
    )


# Build tabs dynamically (Filter out tabs where the *content* is None)
tab_contents = [
    ('pca_tab', get_pca_tab()),
    ('annotation_tab', get_annotation_tab_AgGrid()),
    ('eigenvalues_tab', get_eigenvalues_tab()),
    ('statistics_tab', get_statistics_tab()),
    ('help_tab', get_help_tab())
]
tab_contents = [(tab_id, content) for tab_id, content in tab_contents if content is not None]

# Extract IDs and contents
tab_ids = [tab_id for tab_id, _ in tab_contents]
tab_components = [content for _, content in tab_contents]
tab_outputs = [Output(f"{tab_id}_content", "style") for tab_id in tab_ids]


###---------------------------------------------------


## Dash layout
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = html.Div([
    dcc.Store(id='selected-ids', data=init_selected_ids), # Store all selected ids
    dcc.Store(id='selected-source', data='initial'), # Source of the selection change
    dcc.Store(id='marker-aesthetics-store', data={init_group: init_aesthetics}), # Store marker aesthetics
    dcc.Store(id='trace-pca', data=init_pca_trace), # Store trace for PCA plot
    dcc.Store(id='trace-map', data=init_map_trace), # Store trace for map plot
    dcc.Store(id='trace-time', data=init_time_trace), # Store trace for time plot


    ## hidden div to hold colorscale dropdown
    html.Div([
        dcc.Dropdown(id="colorscale-dropdown", options=[], value=None, style={"display": "none"}),
        dbc.Input(id="default-point-size-modal", type="number", value=args.point_size, style={"display": "none"}),
        dbc.Input(id="default-point-opacity-modal", type="number", value=args.point_opacity, style={"display": "none"}),
        dbc.Input(id="default-point-symbol-modal", type="number", value=args.point_symbol, style={"display": "none"}),
        dcc.Download(id="download-aesthetics"),
    ], style={"display": "none"}),
    html.Div(id="key-modifier", style={"display": "none"}),  # store key modifier state

    ## color modal
    dbc.Modal(
        [
            #dbc.ModalHeader(dbc.ModalTitle("Select Colors")),
            dbc.ModalBody(id="color-modal-body", style={"backgroundColor": "white", "padding": "0px", "height": "100%", "width": "100%"}),
            dbc.ModalFooter(
                html.Div(
                    [
                        dbc.Button("Save", id="save-close-color-modal", n_clicks=0, style={"marginRight": "12px"}),
                        dbc.Button("Cancel", id="cancel-close-color-modal", n_clicks=0, style={"marginRight": "12px"}),
                        dbc.Button("Save to file", id="save-file-color-modal", n_clicks=0, style={"marginLeft": "auto"})
                    ],
                    style={"display": "flex", "width": "100%"}
                )
            ),
        ],
        id="color-modal",
        is_open=False,
        centered=True,
        backdrop=True,
        scrollable=True,
        style={
            "position": "fixed",
            "top": "10%",
            "left": "10%",
            "zIndex": 9999,
            "width": "auto",
            "minWidth": "400px",
            "maxWidth": "90vw",
            "height": "auto",
            "maxHeight": "80vh",
            "display": "flex",
            "flexDirection": "column",
            "boxSizing": "border-box",
            "overflowY": "hidden"
        },
    ), 
    labeled_tabs(tab_ids),
    html.Div(id='tabs-content', children=tab_components)
])   

##----------------------------------------------------

In [ ]:
####################

In [ ]:
## Client side callbacks

## draggable sliders
app.clientside_callback(
    """
    function(n) {
        const container = document.getElementById('split-container');
        const leftPanel = document.getElementById('left-panel');
        const rightPanel = document.getElementById('right-panel');
        const divider = document.getElementById('divider');

        const vLeft = document.getElementById('vertical-divider-left');
        const vRight = document.getElementById('vertical-divider-right');
        const pca = document.getElementById('pca_panel');
        const time = document.getElementById('time-plot-panel');
        const map = document.getElementById('map_panel');
        const table = document.getElementById('data-table-panel');

        if (!container || !divider || !leftPanel || !rightPanel) {
            return window.dash_clientside.no_update;
        }

        // ---- MAIN vertical (left-right) divider ----
        let draggingLR = false;
        divider.onmousedown = (e) => {
            draggingLR = true;
            document.body.style.cursor = 'col-resize';
        };
        window.onmouseup = () => {
            draggingLR = false;
            draggingLeftV = false;
            draggingRightV = false;
            document.body.style.cursor = 'default';
        };
        window.onmousemove = (e) => {
            if (draggingLR) {
                const rect = container.getBoundingClientRect();
                const percent = Math.min(Math.max((e.clientX - rect.left) / rect.width * 100, 10), 90);
                leftPanel.style.flex = `0 0 ${percent}%`;
                rightPanel.style.flex = '1 1 auto';
            }
        };

        // ---- LEFT vertical (PCA-Time) divider ----
        let draggingLeftV = false;
        if (vLeft && pca && time) {
            vLeft.onmousedown = (e) => {
                draggingLeftV = true;
                document.body.style.cursor = 'row-resize';
            };
            window.addEventListener('mousemove', (e) => {
                if (draggingLeftV) {
                    const rect = leftPanel.getBoundingClientRect();
                    const y = e.clientY - rect.top;
                    const percent = Math.min(Math.max((y / rect.height) * 100, 10), 90);
                    pca.style.flex = `1 1 ${percent}%`;
                    time.style.flex = `1 1 ${100 - percent}%`;
                }
            });
        }

        // ---- RIGHT vertical (Map-Table) divider ----
        let draggingRightV = false;
        if (vRight && map && table) {
            vRight.onmousedown = (e) => {
                draggingRightV = true;
                document.body.style.cursor = 'row-resize';
            };
            window.addEventListener('mousemove', (e) => {
                if (draggingRightV) {
                    const rect = rightPanel.getBoundingClientRect();
                    const y = e.clientY - rect.top;
                    const percent = Math.min(Math.max((y / rect.height) * 100, 10), 90);
                    map.style.flex = `1 1 ${percent}%`;
                    table.style.flex = `1 1 ${100 - percent}%`;
                }
            });
        }

        return window.dash_clientside.no_update;
    }
    """,
    Output("init-trigger", "data"),
    Input("init-trigger", "data"),
    prevent_initial_call=False,
)



In [ ]:
## TAB callbacks
# Callback to switch tab content
@app.callback(
    tab_outputs,
    Input("tabs", "value")
)
@log_time("display_tab_content")
def display_tab_content(active_tab):
    return [
        {'display': 'block'} if tab_id == active_tab else {'display': 'none'}
        for tab_id in tab_ids
    ]

### ---------------------------------------------------
## Selected count message update callback (independent, can be called safely)
if ANNOTATION_LAT:
    @app.callback(
        Output("selection-count", "children"),
        Input('selected-ids', 'data'),
        Input('selected-source', 'data')
    )
    @log_time("update_selection_count")
    def update_selection_count(selected_indices, selected_source):
        if not ANNOTATION_LAT:
            raise PreventUpdate

        total = len(df)
        selected = len(selected_indices)
        return f"{selected} of {total} points selected (from {selected_source})"



In [ ]:
## figure update callback

## updates only selected points, not the figure itself
def update_figure_selection_fast(selected_ids, fig, trace_map):
    # Clear all selections fast
    if not selected_ids:
        for trace_idx in trace_map.values():
            fig.data[trace_idx].selectedpoints = None
        return fig

    selected_set = set(selected_ids)

    # Iterate only over relevant traces
    for name, trace_idx in trace_map.items():
        trace = fig.data[trace_idx]
        ids = getattr(trace, "customdata", None)
        if ids is None:
            continue

        # Find indices of selected IDs in this trace
        trace.selectedpoints = [
            j for j, idv in enumerate(ids) if idv in selected_set
        ]

    return fig


def get_selected_ids(selection):
    return [p['customdata'] for p in selection['points'] if 'customdata' in p]

In [ ]:
## PCA plot callbacks
@app.callback(
    Output('pca-plot', 'figure'),
    Output('trace-pca', 'data'),
    Input('selected-ids', 'data'),
    Input('selected-source', 'data'),
    Input('marker-aesthetics-store', "data"),
    Input('x-axis-selector', 'value'),
    Input('y-axis-selector', 'value'),
    Input('z-axis-selector', 'value'),
    Input('group-selector', 'value'),
    Input('pca-plot-type-switch', 'value'),
    Input('show-legend', 'value'),
    Input('pca-plot', 'selectedData'),
    State('pca-plot', 'figure'),
    State('trace-pca', 'data'),
    State('pca-plot-type-switch', 'value'),
)
@log_time("update_pca_plot")
def update_pca_plot(selected_ids, selected_source, marker_aesthetics_store, x_col, y_col, z_col, group, plot_type, show_legend, selected_data, current_fig, current_trace, current_plot_type):
    trigger_name = ctx.triggered[0]['prop_id'].split('.')[0] if ctx.triggered else None

    print(f"   updateddddd_pca_plot(): trigger: {trigger_name}; selected_source: {selected_source}; selectedIDS: {len(selected_ids)})")

    ## hide or show legend
    if trigger_name == 'show-legend':
        current_fig['layout']['showlegend'] = 'show' in show_legend
        return current_fig, dash.no_update

    # user interacted with the plot itself
    if trigger_name == 'pca-plot' and current_plot_type != '3D':
        if not selected_data or 'points' not in selected_data:
            return dash.no_update, dash.no_update
        new_ids = get_selected_ids(selected_data)
        fig = update_figure_selection_fast(new_ids, go.Figure(current_fig), current_trace)
        logging.debug(f"   update_pca_plot(): plot changed by selection in itself ... trigger: {trigger_name}; selected_source: {selected_source}; selectedIDS: {len(new_ids)})")
        return fig, dash.no_update


    ## selection changed elsewhere
    if trigger_name == 'selected-ids' and selected_source != 'pca-plot' and current_plot_type != '3D':
        fig = go.Figure(current_fig)
        fig = update_figure_selection_fast([], fig, current_trace) # clear selection first
        fig = update_figure_selection_fast(selected_ids, fig, current_trace)
        logging.debug(f"   update_pca_plot(): plot changed by selection in other plot ... trigger: {trigger_name}; selected_source: {selected_source}; selectedIDS: {len(selected_ids)})")
        
        # Clear lasso/box overlay from layout selections
        if "selections" in fig.layout:
            fig.layout.selections = []

        return fig, dash.no_update

    ## recreate figure due to layout / aesthetics changes
    pca_triggers = ['marker-aesthetics-store', 'x-axis-selector', 'y-axis-selector', 'z-axis-selector', 'group-selector', 'pca-plot-type-switch', 'selected-ids'] ## add selected-ids for the 3D plot
    if trigger_name in pca_triggers:
        fig, trace_map = generate_pca_fig(dict_of_dicts_to_tuple(marker_aesthetics_store[group]), x_col, y_col, z_col, group, plot_type, tuple(selected_ids))
        if current_plot_type != '3D':
            fig = update_figure_selection_fast(selected_ids, fig, trace_map)
        logging.debug(f"   update_pca_plot(): recreating figure ... trigger: {trigger_name}; selected_source: {selected_source}; selectedIDS: {len(selected_ids)}")
        return fig, trace_map
        
    return dash.no_update, dash.no_update


###---------------------------------------------------
## PCA-plot 2D or 3D switch
@app.callback(
    Output('z-axis-div', 'style'),
    Input('pca-plot-type-switch', 'value'),
)
def toggle_z_axis(plot_type):
    if plot_type == '3D':
        return {'display': 'flex', 'alignItems': 'center', 'marginRight': '20px'}
    else:
        return {'display': 'none'}

In [ ]:
## MAP plot callbacks
if ANNOTATION_LAT:
    @app.callback(
        Output('map-plot', 'figure'),
        Output('trace-map', 'data'),
        Input('selected-ids', 'data'),
        Input('selected-source', 'data'),
        Input('marker-aesthetics-store', 'data'),
        Input('group-selector', 'value'),
        Input('map-plot', 'selectedData'),
        State('map-plot', 'figure'),
        State('trace-map', 'data'),
    )
    @log_time("update_map_plot")
    def update_map_plot(selected_ids, selected_source, marker_aesthetics_store, group, selected_data, current_fig, current_trace):
        trigger_name = ctx.triggered[0]['prop_id'].split('.')[0] if ctx.triggered else None

        if not ANNOTATION_LAT:
            raise PreventUpdate

        # user interacted with the plot itself
        if trigger_name == 'map-plot':
            if not selected_data or 'points' not in selected_data:
                return dash.no_update, dash.no_update
            new_ids = get_selected_ids(selected_data)
            fig = update_figure_selection_fast(new_ids, go.Figure(current_fig), current_trace)
            logging.debug(f"   update_map_plot(): plot changed by selection in itself ... trigger: {trigger_name}; selected_source: {selected_source}; selectedIDS: {len(new_ids)})")
            return fig, dash.no_update


        ## selection changed elsewhere
        if trigger_name == 'selected-ids' and selected_source != 'map-plot':
            fig = go.Figure(current_fig)
            fig = update_figure_selection_fast([], fig, current_trace) # clear selection first
            fig = update_figure_selection_fast(selected_ids, fig, current_trace)
            logging.debug(f"   update_map_plot(): plot changed by selection in other plot ... trigger: {trigger_name}; selected_source: {selected_source}; selectedIDS: {len(selected_ids)})")

            # Clear lasso/box overlay from layout selections
            if "selections" in fig.layout:
                fig.layout.selections = []

            return fig, dash.no_update

        ## recreate figure due to layout / aesthetics changes
        map_triggers = ['marker-aesthetics-store', 'group-selector']
        if trigger_name in map_triggers:
            fig, trace_map = generate_map_fig(dict_of_dicts_to_tuple(marker_aesthetics_store[group]), group)
            fig = update_figure_selection_fast(selected_ids, fig, trace_map)
            logging.debug(f"   update_map_plot(): recreating figure ... trigger: {trigger_name}; selected_source: {selected_source}; selectedIDS: {len(selected_ids)}")
            return fig, trace_map
        
        return dash.no_update, dash.no_update

In [ ]:
## TIME plot callbacks
if ANNOTATION_TIME:
    @app.callback(
        Output('time-plot', 'figure'),
        Output('trace-time', 'data'),
        Input('selected-ids', 'data'),
        Input('selected-source', 'data'),
        Input('marker-aesthetics-store', 'data'),
        Input('group-selector', 'value'),
        Input('time-plot-type-switch', 'value'),
        Input('time-selector', 'value'),
        Input('time-plot', 'selectedData'),
        State('time-plot', 'figure'),
        State('trace-time', 'data'),
    )
    @log_time("update_time_plot")
    def update_time_plot(selected_ids, selected_source, marker_aesthetics_store, group, plot_type, time_variable, selected_data, current_fig, current_trace):
        trigger_name = ctx.triggered[0]['prop_id'].split('.')[0] if ctx.triggered else None

        logging.debug(f"   UPDATE_time_plot(): plot changed by selection in itself ... trigger: {trigger_name}")

        if not init_continuous:
            raise PreventUpdate

        if plot_type != 0 and (trigger_name in ['marker-aesthetics-store', 'group-selector', 'point-size-selector']):
            return dash.no_update, dash.no_update

        if plot_type == 0:
            # user interacted with the plot itself
            if trigger_name == 'time-plot':
                if not selected_data or 'points' not in selected_data:
                    return dash.no_update, dash.no_update
                new_ids = get_selected_ids(selected_data)
                fig = update_figure_selection_fast(new_ids, go.Figure(current_fig), current_trace)
                logging.debug(f"   update_time_plot(): plot changed by selection in itself ... trigger: {trigger_name}; selected_source: {selected_source}; selectedIDS: {len(new_ids)})")
                return fig, dash.no_update


            ## selection changed elsewhere
            if trigger_name == 'selected-ids' and selected_source != 'time-plot':
                fig = go.Figure(current_fig)
                fig = update_figure_selection_fast([], fig, current_trace) # clear selection first
                fig = update_figure_selection_fast(selected_ids, fig, current_trace)
                logging.debug(f"   update_time_plot(): plot changed by selection in other plot ... trigger: {trigger_name}; selected_source: {selected_source}; selectedIDS: {len(selected_ids)})")

                # Clear lasso/box overlay from layout selections
                if "selections" in fig.layout:
                    fig.layout.selections = []

                return fig, dash.no_update

        ## recreate figure due to layout / aesthetics changes
        time_triggers = ['marker-aesthetics-store', 'x-axis-selector', 'y-axis-selector', 'z-axis-selector', 'group-selector', 'time-plot-type-switch', 'selected-ids', 'time-selector']
        if trigger_name in time_triggers:
            fig, trace_time = generate_time_fig(tuple(selected_ids), dict_of_dicts_to_tuple(marker_aesthetics_store[group]), time_variable, group, plot_type)
            fig = update_figure_selection_fast(selected_ids, fig, trace_time)
            logging.debug(f"   update_time_plot(): recreating figure ... trigger: {trigger_name}; selected_source: {selected_source}; selectedIDS: {len(selected_ids)}")
            return fig, trace_time

        return dash.no_update, dash.no_update

In [ ]:
## Data table callbacks
if annotation_desc is not None:
    @app.callback(
        Output('data-table', 'columnDefs'),
        Input('annotation-table', 'cellValueChanged'),
        State('annotation-table', 'rowData'),
        prevent_initial_call=True
    )
    @log_time("update_data_table_columns")
    def update_data_table_columns(cell_event, annotation_rows):

        if annotation_desc is None:
            raise PreventUpdate

        if not cell_event or not annotation_rows:
            return dash.no_update

        # Filter for rows where Selected is True
        selected_rows = [row for row in annotation_rows if row.get('Selected')]

        if not selected_rows:
            return []

        # Extract selected Abbreviations, ensuring 'id' is first
        selected_columns = ['id'] + [col for col in [row['Abbreviation'] for row in selected_rows if 'Abbreviation' in row] if col != 'id']

        # Define column definitions for data table
        defs = get_data_table_column_defs(selected_columns)
        return defs


if annotation_desc is not None:
    @app.callback(
        Output('data-table', 'rowData'),
        Input('selected-ids', 'data'),
        Input('selected-source', 'data'),
        State('data-table', 'rowData'),
        prevent_initial_call=False
    )
    @log_time("update_table_data")
    def update_table_selected(selected_ids, selected_source, current_data):
        if selected_source == 'data-table' or not current_data:
            return dash.no_update
        
        df_current = pd.DataFrame(current_data)
        current_ids = df_current.loc[df_current['Selected'], 'id'].tolist()

        if set(selected_ids) == set(current_ids):
            return dash.no_update
        if not selected_ids:
            return []
        
        # Update the 'Selected' column
        df_current['Selected'] = df_current['id'].isin(selected_ids)
        return df_current.to_dict('records')

In [ ]:
## dcc.Store marker_aesthetics callbacks
from turtle import color

## create the updated aesthetics store for the current group
def new_marker_aesthetics_store(
        cur_store, 
        col_type, 
        color_values,  color_ids, 
        size_values, size_ids, 
        opacity_values, opacity_ids, 
        symbol_values, symbol_ids, 
        group, 
        colorscale, 
        default_point_size_modal, 
        default_point_opacity_modal, 
        default_point_symbol_modal):
    new_store = copy.deepcopy(cur_store) # copy() is only copying the top level: not sufficient

    logging.debug(f"new_marker_aesthetics_store(): size_values={size_values}, size_ids={size_ids}, symbol_values={symbol_values}, symbol_ids={symbol_ids}")

    ## size default
    if default_point_size_modal:
        new_store['size']['default'] = default_point_size_modal

    ## opacity default
    if default_point_opacity_modal:
        new_store['opacity']['default'] = default_point_opacity_modal

    ## symbol default
    if default_point_symbol_modal:
        new_store['symbol']['default'] = default_point_symbol_modal
        new_store['symbol_map']['default'] = default_point_symbol_modal if default_point_symbol_modal in maplot_point_symbols else 'circle'

    ## color
    if col_type in 'fi':
        assert(colorscale is not None), "Colorscale must be selected for continuous variable."
        new_store['color'] = {
            'colorscale': colorscale,
            'cmin': df[group].min(),
            'cmax': df[group].max(),
            'nan_color': "#0d08085d"
        }
    else:
        ## color
        if color_values and color_ids:
            ## color: store all
            for item, val in zip(color_ids, color_values):
                new_store['color'][item["index"]] = val

        ## point size
        if size_values and size_ids:
            ## size: store only the ones deviating from default
            for item, val in zip(size_ids, size_values):
                if val != '-' and val != default_point_size_modal:
                    new_store['size'][item["index"]] = val

        ## opacity
        if opacity_values and opacity_ids:
            ## opacity: store only the ones deviating from default
            for item, val in zip(opacity_ids, opacity_values):
                if val != '-' and val != default_point_opacity_modal:
                    new_store['opacity'][item["index"]] = val

        ## symbol symbol
        if symbol_values and symbol_ids:
            ## symbol: store only the ones deviating from default
            for item, val in zip(symbol_ids, symbol_values):
                if val != '-' and val != default_point_symbol_modal:
                    new_store['symbol'][item["index"]] = val
                    new_store['symbol_map'][item["index"]] = val if val in maplot_point_symbols else new_store['symbol_map']['default']

    return new_store


## Callback to update marker aesthetics store
## 'color' is always present with all keys for categorical variables, or 'colorscale' for continuous variables
## 'size' is always present, and contains at least 'default'
## 'symbol' is always present, and contains at least 'default'
@app.callback(
    Output("color-modal", "is_open"),
    Output("color-modal-body", "children"),
    Output("marker-aesthetics-store", "data"),
    Output("download-aesthetics", "data"),
    Input("group-selector", "value"),
    Input("show-table-btn", "n_clicks"),
    Input("save-close-color-modal", "n_clicks"),
    Input("cancel-close-color-modal", "n_clicks"),
    Input("save-file-color-modal", "n_clicks"),
    State("marker-aesthetics-store", "data"),
    State("color-modal", "is_open"),
    State({"type": "color-picker-modal", "index": ALL}, "value"),
    State({"type": "color-picker-modal", "index": ALL}, "id"),
    State({"type": "size-picker-modal", "index": ALL}, "value"),
    State({"type": "size-picker-modal", "index": ALL}, "id"),
    State({"type": "opacity-picker-modal", "index": ALL}, "value"),
    State({"type": "opacity-picker-modal", "index": ALL}, "id"),
    State({"type": "symbol-picker-modal", "index": ALL}, "value"),
    State({"type": "symbol-picker-modal", "index": ALL}, "id"),
    State("colorscale-dropdown", "value"),
    State("default-point-size-modal", "value"),
    State("default-point-opacity-modal", "value"),
    State("default-point-symbol-modal", "value"),
)
@log_time("update_marker_aesthetics_store")
def update_marker_aesthetics_store(
    group,
    n_show_table_clicks,
    n_save_close_clicks,
    n_cancel_close_clicks,
    n_save_file_clicks,
    current_marker_aesthetics_store,
    is_open,
    color_values, color_ids,
    size_values, size_ids,
    opacity_values, opacity_ids,
    symbol_values, symbol_ids,
    colorscale,
    default_point_size_modal,
    default_point_opacity_modal,
    default_point_symbol_modal,
    n_columns=1
):
    import dash
    trigger_name = ctx.triggered[0]['prop_id'].split('.')[0] if ctx.triggered else None
    logging.debug(f"update_color_store triggered by {trigger_name}, group={group}, is_open={is_open}, current_marker_aesthetics_store={current_marker_aesthetics_store}")

    # When group changes, get the appropriate color map or create the default one: initialization
    if (trigger_name == 'group-selector') or not trigger_name:
        if group not in current_marker_aesthetics_store:
            current_marker_aesthetics_store[group] = get_init_aesthetics_of_group(group)
        logging.debug(f"Group changed to {group}, resetting color store: {current_marker_aesthetics_store[group]}")
        return dash.no_update, dash.no_update, current_marker_aesthetics_store, dash.no_update

    ## get the store for the current group
    assert(isinstance(current_marker_aesthetics_store, dict) and group in current_marker_aesthetics_store), f"Group {group} not found in current_marker_aesthetics_store: {current_marker_aesthetics_store}"
    assert('color' in current_marker_aesthetics_store[group]), f"'color' key not found in current_marker_aesthetics_store for group {group}: {current_marker_aesthetics_store[group]}"
    assert('size' in current_marker_aesthetics_store[group]), f"'size' key not found in current_marker_aesthetics_store for group {group}: {current_marker_aesthetics_store[group]}"
    group_store = current_marker_aesthetics_store[group]

    # Get column type (continuous or categorical)
    col_type = df[group].dtype.kind


    ##---------------------------------------
    # Close modal without saving
    if trigger_name == 'cancel-close-color-modal' or (trigger_name == 'show-table-btn' and is_open):
        logging.debug("Closing color modal without saving.")
        return False, dash.no_update, dash.no_update, dash.no_update


    ##---------------------------------------
    ## Save to file modal: update color-map
    if trigger_name == 'save-file-color-modal':
        new_store = new_marker_aesthetics_store(group_store, col_type, color_values, color_ids, size_values, size_ids, opacity_values, opacity_ids, symbol_values, symbol_ids, group, colorscale, default_point_size_modal, default_point_opacity_modal, default_point_symbol_modal)
        current_marker_aesthetics_store[group] = new_store

        logging.debug("Downloaded aesthetics store to file.")
        return (dash.no_update, dash.no_update, dash.no_update, 
                {"content": json.dumps(current_marker_aesthetics_store, indent=2), "filename": "interactivePCA_aesthetics.json", "type": "application/json"})


    ##---------------------------------------
    ## Save and close modal: update color-map
    if trigger_name == 'save-close-color-modal':
        new_store = new_marker_aesthetics_store(group_store, col_type, color_values, color_ids, size_values, size_ids, opacity_values, opacity_ids, symbol_values, symbol_ids, group, colorscale, default_point_size_modal, default_point_opacity_modal, default_point_symbol_modal)

        # Do not update if not changed
        if new_store == group_store:
            logging.debug(f"No update of color map as it did not change.")
            return False, dash.no_update, dash.no_update, dash.no_update

        logging.debug(f"Updated color map: {new_store}")
        current_marker_aesthetics_store[group] = new_store
        return False, dash.no_update, current_marker_aesthetics_store, dash.no_update

    ##---------------------------------------
    # Open modal and populate the table
    if col_type in 'fi':
        assert('colorscale' in group_store['color']), f"Expected current_marker_aesthetics_store to be a dict with 'colorscale' key for continuous variable, got: {group_store['color']}"

        current_color = group_store['color']['colorscale']
        logging.debug(f"Preparing colorscale preview for colorscale: {current_color}")
        dropdown = dcc.Dropdown(
            id="colorscale-dropdown",
            options=[{"label": cs, "value": cs} for cs in px.colors.named_colorscales()],
            value=current_color,
            clearable=False,
            style={"width": "300px", "marginBottom": "16px"}
        )


        # unselected controls for the group (global unselected editable in modal)
        unselected_color_input = dbc.Input(
            type="color",
            id="unselected-point-color-modal",
            value=group_store.get('color', {}).get('unselected', '#cccccc'),
            style={"width": "40px", "height": "30px", "marginRight": "12px"}
        )
        unselected_size_input = dbc.Input(
            type="number",
            id="unselected-point-size-modal",
            min=1,
            max=40,
            step=1,
            value=group_store.get('size', {}).get('unselected', args.point_size),
            style={"width": "80px", "height": "32px", "marginRight": "12px"}
        )
        unselected_opacity_input = dbc.Input(
            type="number",
            id="unselected-point-opacity-modal",
            min=0,
            max=1,
            step=0.05,
            value=group_store.get('opacity', {}).get('unselected', args.point_size_unselected),
            style={"width": "80px", "height": "32px", "marginRight": "12px"}
        )
        unselected_symbol_input = dcc.Dropdown(
            id="unselected-point-symbol-modal",
            options=[{"label": s, "value": s} for s in plotly_point_symbols],
            value=group_store.get('symbol', {}).get('unselected', plotly_point_symbols[0]),
            clearable=False,
            style={"width": "150px", "height": "32px", "marginRight": "12px"}
        )
        unselected_row = html.Div([
            html.Div("Unselected:", style={"fontWeight": "bold", "marginBottom": "6px"}),
            html.Div(["Color: ", unselected_color_input], style={"display": "inline-flex", "alignItems": "center", "marginRight": "16px"}),
            html.Div(["Size: ", unselected_size_input], style={"display": "inline-flex", "alignItems": "center", "marginRight": "16px"}),
            html.Div(["Opacity: ", unselected_opacity_input], style={"display": "inline-flex", "alignItems": "center", "marginRight": "16px"}),
            html.Div(["Symbol: ", unselected_symbol_input], style={"display": "inline-flex", "alignItems": "center"}),
        ], style={"marginBottom": "12px"})

        preview = html.Div([
            html.Div("Select colorscale for continuous variable:", style={"marginBottom": "8px"}),
            dropdown,
        ])
        logging.debug(f"Opening color modal with colorscale dropdown for group {group}.")
        return True, preview, dash.no_update, dash.no_update

    else:
        # Categorical: show color and size fields
        logging.info("Opening ctegorical")

        default_size = group_store['size']['default']
        default_opacity = group_store['opacity']['default']
        default_symbol = group_store['symbol']['default']

        # Count how many actual categories we have besides default/unselected
        category_keys = [k for k in group_store['color'].keys() if k not in ('default', 'unselected')]
        remove_default_color = len(category_keys) > 0  # True if there are other categories


        pairs = [
            (
                # Color
                dbc.Input(
                    type="color",
                    id={"type": "color-picker-modal", "index": str(key)},
                    value=color_val,
                    style={"width": "40px", "height": "30px", "padding": "0", "border": "1px solid #ced4da"}
                ) if not (key == "default" and remove_default_color) else None,
                # Size
                dcc.Dropdown(
                    id={"type": "size-picker-modal", "index": str(key)},
                    options=[{"label": "-", "value": "-"}] + [{"label": s, "value": s} for s in plotly_point_sizes],
                    value=(
                        "-"
                        if group_store["size"].get(key, default_size) == default_size and key != "default"
                        else group_store["size"].get(key, default_size)
                    ),
                    clearable=False,
                    style={"width": "50px", "height": "30px"}
                ),
                # Opacity
                dcc.Dropdown(
                    id={"type": "opacity-picker-modal", "index": str(key)},
                    options=[{"label": "-", "value": "-"}] + [{"label": round(s,1), "value": s} for s in plotly_opacity_sizes],
                    value=(
                        "-"
                        if group_store["opacity"].get(key, default_opacity) == default_opacity and key != "default"
                        else group_store["opacity"].get(key, default_opacity)
                    ),
                    clearable=False,
                    style={"width": "60px", "height": "30px"}
                ),
                # Symbol
                dcc.Dropdown(
                    id={"type": "symbol-picker-modal", "index": str(key)},
                    options=[{"label": "-", "value": "-"}] + [{"label": f"{s}*" if s not in maplot_point_symbols else s, "value": s} for s in plotly_point_symbols],
                    value=(
                        "-"
                        if group_store["symbol"].get(key, default_symbol) == default_symbol and key != "default"
                        else group_store["symbol"].get(key, default_symbol)
                    ),
                    clearable=False,
                    style={"width": "150px", "height": "30px"}
                ),
                # Category label
                str(key)
            )
            for key, color_val in group_store["color"].items()
        ]

        # Table header (each property gets its own <th>)
        header_cells = []
        for _ in range(n_columns):
            header_cells += [
                html.Th("Color", style={"textAlign": "center"}),
                html.Th("Size", style={"textAlign": "center"}),
                html.Th("Opacity", style={"textAlign": "center"}),
                html.Th("Symbol", style={"textAlign": "center"}),
                html.Th(group, style={"textAlign": "left"})
            ]
        header = html.Tr(header_cells)

        # Table rows
        grid_rows = []
        for row_idx, i in enumerate(range(0, len(pairs), n_columns)):
            row_cells = []
            for color_input, size_input, opacity_input, symbol_input, cat in pairs[i:i+n_columns]:
                # Wrap each control in a flex div for vertical alignment
                row_cells.append(html.Td(html.Div(color_input, style={"display":"flex", "alignItems":"center"})))
                row_cells.append(html.Td(html.Div(size_input, style={"display":"flex", "alignItems":"center"})))
                row_cells.append(html.Td(html.Div(opacity_input, style={"display":"flex", "alignItems":"center"})))
                row_cells.append(html.Td(html.Div(symbol_input, style={"display":"flex", "alignItems":"center"})))
                row_cells.append(html.Td(html.Div(str(cat), style={"paddingRight": "16px", "display":"flex", "alignItems":"center"})))

            grid_rows.append(html.Tr(row_cells))

            # Add separation space after "unselected" row
            if row_idx == 1:
                grid_rows.append(
                    html.Tr([
                        html.Td(
                            "",  # empty cell
                            colSpan=5 * n_columns,  # spans full width
                            style={"height": "16px"}  # adjust spacing here
                        )
                    ])
                )


        table = html.Div([
            html.Table(
                [header] + grid_rows,
                style={"borderCollapse": "collapse", "width": "100%"}
            ),
            html.Div("An asterisk (*) indicates that this symbol is not available in the map plot and will be replaced with the default symbol (or a circle) in the map plot.", style={"marginTop": "15px"}),
        ])

        table_div = html.Div(
            [table],
            style={
                "overflowX": "auto",
                "overflowY": "auto",
                "textAlign": "left",
                "display": "flex",
                "flexDirection": "column",
                "width": "100%",
                "height": "100%",
                "maxWidth": "100vw",
                "maxHeight": "70vh",
                "verticalAlign": "top",
                "boxSizing": "border-box",
                "padding": "32px"
            }
        )

        logging.debug(f"Opening color modal with table for group {group}.")
        return True, table_div, dash.no_update, dash.no_update

In [ ]:
## dcc.Store selected-ids callback
from turtle import mode


## get dynamic input specs
input_specs = []

if annotation_desc is not None:
    input_specs.append(("texarea_query", Input("texarea-expression", "value")))
    input_specs.append(("table_selection", Input("data-table", "cellValueChanged")))

if ANNOTATION_TIME:
    input_specs.append(("time_relayout", Input("time-plot", "relayoutData")))
    input_specs.append(("time_selection", Input("time-plot", "selectedData")))

input_specs.append(("pca_selection", Input("pca-plot", "selectedData")))

if ANNOTATION_LAT:
    input_specs.append(("map_selection", Input("map-plot", "selectedData")))

input_specs.append(("cur_selected_ids", State("selected-ids", "data")))

if annotation_desc is not None:
    input_specs.append(("cur_data_table", State("data-table", "rowData")))

input_specs.append(("cur_selected_source", State("selected-source", "data")))


inputs = [i for _, i in input_specs]
input_names = [n for n, _ in input_specs]


@app.callback(
    Output('selected-ids', 'data'),
    Output('selected-source', 'data'),
    *inputs,    
    prevent_initial_call=True
)
@log_time("update_selected_indices")
def update_selected_indices(*args):
    # Unpack dynamically
    args_dict = dict(zip(input_names, args))


    trigger_name = ctx.triggered[0]['prop_id'].split('.')[0] if ctx.triggered else None


    if trigger_name is None:
        return dash.no_update, dash.no_update
    

    ## pca-plot
    if trigger_name == "pca-plot":
        if not args_dict.get('pca_selection') or 'points' not in args_dict.get('pca_selection', {}):
            return dash.no_update, dash.no_update
        new_selected_ids = get_selected_ids(args_dict['pca_selection'])
        logging.debug(f"   update_selected_indices(): PCA plot selection updated: new_selected_ids: {len(new_selected_ids)}; map_selected_ids: {args_dict.get('map_selection')}; trigger_name: {trigger_name}; selected_source: {args_dict['cur_selected_source']}")
        return new_selected_ids, 'pca-plot'


    ## map-plot
    if trigger_name == "map-plot":
        if not args_dict.get('map_selection') or 'points' not in args_dict.get('map_selection', {}):
            return dash.no_update, dash.no_update
        new_selected_ids = get_selected_ids(args_dict['map_selection'])
        logging.debug(f"   update_selected_indices(): Map plot selection updated: new_selected_ids: {len(new_selected_ids)}; pca_selected_ids: {args_dict.get('pca_selection')}; trigger_name: {trigger_name}; selected_source: {args_dict['cur_selected_source']}")
        return new_selected_ids, 'map-plot'
    

    ## time-plot selectedData
    if trigger_name == "time-plot" and args_dict.get('time_selection'):
        new_selected_ids = get_selected_ids(args_dict['time_selection'])
        logging.debug(f"   update_selected_indices(): Time plot selection updated: new_selected_ids: {len(new_selected_ids)}; pca_selected_ids: {args_dict.get('pca_selection')}; trigger_name: {trigger_name}; selected_source: {args_dict['cur_selected_source']}")
        return new_selected_ids, 'time-plot'


    ## time-plot relayoutData
    if trigger_name == "time-plot":
        if ANNOTATION_TIME is None:
            return dash.no_update, dash.no_update
        if not args_dict.get('time_relayout'):
            return dash.no_update, dash.no_update

        x_range = args_dict['time_relayout'].get('xaxis.range', None)
        if x_range is None:
            x0 = args_dict['time_relayout'].get('xaxis.range[0]')
            x1 = args_dict['time_relayout'].get('xaxis.range[1]')
            if x0 is not None and x1 is not None:
                x_range = [x0, x1]
        if x_range:
            filtered_df = df[(df[ANNOTATION_TIME] >= x_range[0]) & (df[ANNOTATION_TIME] <= x_range[1])]
            new_ids = filtered_df['id'].tolist()
        else:
            new_ids = df['id'].tolist()

        if set(new_ids) == set(args_dict['cur_selected_ids']):
            return dash.no_update, dash.no_update
        
        return new_ids, 'time-plot'
    

    ## texarea-expression
    if trigger_name == "texarea-expression":
        texarea_query = args_dict.get('texarea_query')
        if not texarea_query or str(texarea_query).strip() == "":
            return dash.no_update, dash.no_update
        
        try:
            filtered_df = df.query(texarea_query)
            #logging.debug(f"Filtered {len(filtered_df)} rows using query: {texarea_query}")
            return filtered_df['id'].tolist(), 'texarea-expression'
        except Exception as e:
            return dash.no_update, dash.no_update

    
    ## data-table
    if trigger_name == "data-table":
        cur_data_table = args_dict.get('cur_data_table')
        if not cur_data_table:
            return dash.no_update, dash.no_update

        df_current = pd.DataFrame(cur_data_table)
        selected_ids = df_current.loc[df_current["Selected"], "id"].tolist()

        if set(selected_ids) == set(args_dict['cur_selected_ids']):
            return dash.no_update, dash.no_update
        return selected_ids, 'data-table'


    ## fallback
    return dash.no_update, dash.no_update

In [ ]:
####################

In [ ]:
## launch dash server
if __name__ == '__main__':
    url = f"http://localhost:{args.server_port}"
    logging.info(f"Dashboard running at {url}")
    if args.open_browser:
        webbrowser.open(url)
    app.run(debug=args.dev, port=args.server_port)

## to kill a running isntance
## lsof -i :8050 | awk 'NR!=1 {print $2}' | xargs kill -9